# 🏥 Advanced Medical RAG Chatbot — Google Colab GPU

- BGE-M3 và Cross-Encoder chạy trên GPU Colab.
- ChromaDB hoạt động trong `/content`.



## 1. Cài đặt thư viện và môi trường local


In [3]:
# Colab đã có PyTorch CUDA, không cài lại torch.
%pip install -q pandas pyarrow chromadb sentence-transformers rank_bm25 tavily-python tqdm nest-asyncio groq


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 115.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 129.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 122.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60

In [4]:
import os
import json
import hashlib
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import nest_asyncio
nest_asyncio.apply()
import unicodedata
from google.colab import userdata

def get_secret(name: str, default: str = "") -> str:
    try:
        value = userdata.get(name)
        return value.strip() if value else default
    except Exception:
        return default

GROQ_API_KEY = get_secret("GROQ_API_KEY")
TAVILY_API_KEY = get_secret("TAVILY_API_KEY")

from groq import Groq
from tavily import TavilyClient

groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

FAST_MODEL = "openai/gpt-oss-20b"
STRONG_MODEL = "qwen/qwen3-32b"

AUXILIARY_TASKS = {
    "query_rewrite",
    "spell_correction",
    "hyde_generation",
    "crag_grading",
}

TASK_MAX_TOKENS = {
    "query_rewrite": 256,
    "spell_correction": 256,
    "hyde_generation": 256,
    "crag_grading": 256,
    "final_generation": 2048,
    "general": 1024,
}

MODEL_DISABLED_UNTIL = {
    FAST_MODEL: 0.0,
    STRONG_MODEL: 0.0,
}

LLM_METRICS = {
    "logical_calls": 0,
    "requests_by_model": {
        FAST_MODEL: 0,
        STRONG_MODEL: 0,
    },
    "failures_by_model": {
        FAST_MODEL: 0,
        STRONG_MODEL: 0,
    },
    "calls_by_task": {},
}

if groq_client:
    print("[OK] Groq Client đã sẵn sàng.")
    print(f"     - FAST/AUXILIARY: {FAST_MODEL}")
    print(f"     - STRONG/FINAL:   {STRONG_MODEL}")
else:
    print("[WARN] Chưa có GROQ_API_KEY trong Colab Secrets.")


def is_valid_llm_output(value):
    return (
        isinstance(value, str)
        and bool(value.strip())
        and not value.strip().startswith("[LỖI]")
    )


def _is_rate_limit_error(error: Exception) -> bool:
    text = str(error).lower()
    status_code = getattr(error, "status_code", None)
    return (
        status_code == 429
        or "429" in text
        or "rate_limit" in text
        or "rate limit" in text
    )


def _model_order_for_task(task_name: str):
    if task_name in AUXILIARY_TASKS:
        return [FAST_MODEL, STRONG_MODEL]
    return [STRONG_MODEL, FAST_MODEL]


def _call_groq_model(
    model: str,
    prompt: str,
    system_instruction: str = None,
    json_mode: bool = False,
    temperature: float = 0.0,
    max_tokens: int = 1024,
) -> str:
    messages = []

    if system_instruction:
        messages.append({
            "role": "system",
            "content": system_instruction,
        })

    messages.append({
        "role": "user",
        "content": prompt,
    })

    kwargs = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    if json_mode:
        kwargs["response_format"] = {"type": "json_object"}

    if model == STRONG_MODEL:
        kwargs["reasoning_effort"] = "none"

    try:
        response = groq_client.chat.completions.create(**kwargs)
    except TypeError:
        # Tương thích SDK/provider không nhận reasoning_effort.
        kwargs.pop("reasoning_effort", None)
        response = groq_client.chat.completions.create(**kwargs)

    return response.choices[0].message.content.strip()


def call_llm(
    prompt: str,
    system_instruction: str = None,
    json_mode: bool = False,
    temperature: float = 0.0,
    task_name: str = "general",
):
    if not groq_client:
        print("      ❌ GROQ_API_KEY chưa được thiết lập.")
        return None

    LLM_METRICS["logical_calls"] += 1

    task_counts = LLM_METRICS["calls_by_task"]
    task_counts[task_name] = task_counts.get(task_name, 0) + 1

    max_tokens = TASK_MAX_TOKENS.get(
        task_name,
        TASK_MAX_TOKENS["general"],
    )

    for model in _model_order_for_task(task_name):
        now = time.time()

        if now < MODEL_DISABLED_UNTIL.get(model, 0):
            remaining = int(MODEL_DISABLED_UNTIL[model] - now)
            print(
                f"      ⏭️ Bỏ qua {model}: "
                f"đang cooldown khoảng {remaining}s."
            )
            continue

        try:
            LLM_METRICS["requests_by_model"][model] += 1

            result = _call_groq_model(
                model=model,
                prompt=prompt,
                system_instruction=system_instruction,
                json_mode=json_mode,
                temperature=temperature,
                max_tokens=max_tokens,
            )

            if is_valid_llm_output(result):
                return result.strip()

        except Exception as e:
            LLM_METRICS["failures_by_model"][model] += 1

            if _is_rate_limit_error(e):
                MODEL_DISABLED_UNTIL[model] = time.time() + 60
                print(
                    f"      ⚠️ [{model}] bị rate-limit; "
                    "tạm bỏ qua trong 60 giây."
                )
            else:
                print(f"      ⚠️ [{model}] lỗi: {e}")

            print("      → Chuyển sang model fallback...")

    print("      ❌ Cả hai model Groq đều thất bại.")
    return None


def show_llm_metrics():
    print(json.dumps(
        LLM_METRICS,
        ensure_ascii=False,
        indent=2,
    ))


[OK] Groq Client đã sẵn sàng.
     - FAST/AUXILIARY: openai/gpt-oss-20b
     - STRONG/FINAL:   qwen/qwen3-32b


## 2. Khởi tạo Cache & Conversation Memory

Hai module mới giúp giảm API calls và hỗ trợ hội thoại đa lượt.

In [5]:
class RAGCache:
    """Cache 3 tầng cho RAG pipeline: Spell, HyDE, Full Answer."""

    def __init__(self, ttl_seconds=3600):
        self.ttl = ttl_seconds
        self.spell_cache = {}
        self.hyde_cache = {}
        self.answer_cache = {}

    def _hash(self, text: str) -> str:
        normalized = str(text).strip().lower()
        return hashlib.md5(normalized.encode("utf-8")).hexdigest()

    def _is_expired(self, entry: dict) -> bool:
        return (time.time() - entry["timestamp"]) > self.ttl

    def get(self, cache_dict: dict, key: str):
        h = self._hash(key)
        entry = cache_dict.get(h)
        if entry is None:
            return None

        if self._is_expired(entry):
            del cache_dict[h]
            return None

        return entry["value"]

    def set(self, cache_dict: dict, key: str, value):
        h = self._hash(key)
        cache_dict[h] = {
            "value": value,
            "timestamp": time.time()
        }

    def clear(self):
        self.spell_cache.clear()
        self.hyde_cache.clear()
        self.answer_cache.clear()

    def stats(self):
        return {
            "spell": len(self.spell_cache),
            "hyde": len(self.hyde_cache),
            "answer": len(self.answer_cache),
        }


class ConversationMemory:
    """Sliding-window memory, dùng riêng cho từng phiên người dùng."""

    def __init__(self, max_turns=5):
        self.history = []
        self.max_turns = max_turns

    def add(self, role: str, content: str):
        if role not in {"user", "bot"}:
            raise ValueError("role phải là 'user' hoặc 'bot'")

        content = str(content).strip()
        if not content:
            return

        self.history.append({"role": role, "content": content})
        self.history = self.history[-self.max_turns * 2:]

    def get_context_string(self, max_chars_per_message=500) -> str:
        lines = []
        for msg in self.history:
            prefix = "Người dùng" if msg["role"] == "user" else "Chatbot"
            content = msg["content"][:max_chars_per_message]
            lines.append(f"{prefix}: {content}")
        return "\n".join(lines)

    def clear(self):
        self.history.clear()


cache = RAGCache(ttl_seconds=3600)

# Mỗi session_id có một ConversationMemory riêng.
session_memories = {}


def get_memory(session_id: str = "default") -> ConversationMemory:
    session_id = str(session_id or "default")
    if session_id not in session_memories:
        session_memories[session_id] = ConversationMemory(max_turns=5)
    return session_memories[session_id]


def clear_session_memory(session_id: str = "default"):
    get_memory(session_id).clear()



import re


# Chỉ những câu có dấu hiệu thiếu chủ thể mới cần dùng lịch sử.
FOLLOW_UP_PATTERNS = [
    # Các từ nối thể hiện đang hỏi tiếp
    r"^còn\b",
    r"^vậy\b",
    r"^thế\b",
    r"^thế còn\b",
    r"^vậy còn\b",
    r"^sau đó\b",
    r"^tiếp theo\b",

    # Câu hỏi ngắn thường bị lược bỏ bệnh/đối tượng
    r"^uống thuốc gì\b",
    r"^dùng thuốc gì\b",
    r"^thuốc nào\b",
    r"^điều trị thế nào\b",
    r"^chữa thế nào\b",
    r"^phòng ngừa thế nào\b",

    # Hỏi tiếp về thuốc hoặc phương pháp vừa nói
    r"^có tác dụng phụ\b",
    r"^tác dụng phụ là gì\b",
    r"^có chống chỉ định\b",
    r"^dùng bao lâu\b",
    r"^liều lượng thế nào\b",

    # Câu thiếu chủ thể
    r"^có nguy hiểm không\b",
    r"^có hại không\b",
    r"^có chữa được không\b",
    r"^có lây không\b",
    r"^có tái phát không\b",

    # Đại từ tham chiếu
    r"\bnó\b",
    r"\bthuốc đó\b",
    r"\bthuốc này\b",
    r"\bcác thuốc đó\b",
    r"\bbệnh đó\b",
    r"\bbệnh này\b",
    r"\btình trạng đó\b",
    r"\btình trạng này\b",
    r"\bphương pháp đó\b",
    r"\bphương pháp này\b",
]


def needs_history_rewrite(query: str) -> bool:
    """
    Chỉ trả về True khi câu hiện tại phụ thuộc vào lịch sử.

    Không dùng quy tắc 'câu ngắn thì rewrite', vì các câu như:
    - Hiện tượng ngủ mở mắt
    - Huyết áp thấp
    - Căng cơ đùi
    tuy ngắn nhưng vẫn là query độc lập.
    """
    query = str(query).strip().lower()

    if not query:
        return False

    query = re.sub(
        r"\s+",
        " ",
        query
    )

    return any(
        re.search(pattern, query)
        for pattern in FOLLOW_UP_PATTERNS
    )


QUERY_REWRITE_SYSTEM_PROMPT = """
Bạn là bộ giải quyết tham chiếu cho hệ thống RAG y tế nhiều lượt.

Nhiệm vụ:
Chỉ khi câu hỏi hiện tại bị thiếu chủ thể, hãy bổ sung chủ thể đó
từ lịch sử hội thoại để tạo thành một câu hỏi độc lập.

Quy tắc bắt buộc:

1. Không trả lời câu hỏi.
2. Chỉ trả về đúng một câu hỏi độc lập.
3. Viết lại ở mức tối thiểu cần thiết.
4. Chỉ bổ sung đối tượng bị lược bỏ, chẳng hạn:
   - tên bệnh;
   - tên thuốc hoặc nhóm thuốc;
   - phương pháp điều trị;
   - triệu chứng;
   - bộ phận cơ thể.
5. Không thêm kiến thức y tế mới.
6. Không thêm nguyên nhân, triệu chứng, tác hại hoặc chẩn đoán
   nếu người dùng không nói trong câu hỏi hiện tại.
7. Không lấy các nhận định trong câu trả lời trước để mở rộng câu hỏi.
8. Không tự tạo tên thuốc hoặc bệnh.
9. Nếu không xác định chắc chắn chủ thể, giữ nguyên câu hỏi.
10. Không thêm Markdown, giải thích hoặc dấu ngoặc kép.

Ví dụ 1:

Lịch sử:
Người dùng: Đau dạ dày nên điều trị thế nào?
Chatbot: Một số nhóm thuốc có thể được sử dụng tùy nguyên nhân.
Người dùng: Uống thuốc gì?

Kết quả:
Uống thuốc gì để điều trị đau dạ dày?

Ví dụ 2:

Lịch sử:
Người dùng: Uống thuốc gì để điều trị đau dạ dày?
Chatbot: Một số nhóm thuốc thường được sử dụng...
Người dùng: Có tác dụng phụ không?

Kết quả:
Các thuốc dùng để điều trị đau dạ dày có tác dụng phụ gì?

Ví dụ 3:

Lịch sử:
Chatbot: Ngủ mở mắt có thể ảnh hưởng đến bề mặt mắt.
Người dùng: Hiện tượng ngủ mở mắt có hại gì không?

Kết quả:
Hiện tượng ngủ mở mắt có hại gì không?

Không được viết thành:
"Hiện tượng ngủ mở mắt có thể liên quan đến thị giác hoặc cấu trúc
cơ mắt, nhưng liệu nó có hại không?"

Lý do:
Câu đó tự thêm thông tin không có trong câu hỏi của người dùng.
""".strip()


def rewrite_query_with_history(
    current_query: str,
    memory: ConversationMemory
) -> str:
    current_query = str(current_query).strip()

    if not current_query:
        return current_query

    history_str = memory.get_context_string()

    # Không có lịch sử thì không cần rewrite.
    if not history_str:
        return current_query

    # Câu đã độc lập thì không gọi LLM.
    if not needs_history_rewrite(current_query):
        return current_query

    prompt = f"""
LỊCH SỬ HỘI THOẠI:
{history_str}

CÂU HỎI HIỆN TẠI:
{current_query}

Hãy chỉ bổ sung chủ thể bị lược bỏ và trả về một câu hỏi độc lập.
""".strip()

    rewritten = call_llm(
        prompt,
        system_instruction=QUERY_REWRITE_SYSTEM_PROMPT,
        temperature=0.0,
        task_name="query_rewrite"
    )

    if not is_valid_llm_output(rewritten):
        return current_query

    rewritten = rewritten.strip()

    # Phòng trường hợp model trả output quá dài hoặc trả lời thay vì rewrite.
    if len(rewritten) > max(300, len(current_query) * 5):
        print(
            "      [WARN] Query rewrite dài bất thường; "
            "giữ nguyên câu hỏi gốc."
        )
        return current_query

    return rewritten


## 3. Chuẩn bị dữ liệu và ChromaDB local

- **DEV_MODE**: index một phần dữ liệu để kiểm thử nhanh.
- **Production build**: đặt `RAG_DEV_MODE=false` và chạy build một lần.
- Index được lưu trong `storage/dev` hoặc `storage/prod`.


In [6]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount(
    "/content/drive",
    force_remount=False
)

DRIVE_ZIP = Path(
    "/content/drive/MyDrive/"
    "medical_rag_backups/"
    "medical_rag_prod.zip"
)

LOCAL_PROD_DIR = Path(
    "/content/medical_rag_runtime/storage/prod"
)

if not DRIVE_ZIP.exists():
    raise FileNotFoundError(
        f"Không tìm thấy backup: {DRIVE_ZIP}"
    )

# Xóa bản local cũ hoặc dở dang.
if LOCAL_PROD_DIR.exists():
    shutil.rmtree(
        LOCAL_PROD_DIR
    )

LOCAL_PROD_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("[INFO] Đang restore full index từ Google Drive...")

shutil.unpack_archive(
    filename=str(DRIVE_ZIP),
    extract_dir=str(LOCAL_PROD_DIR)
)

required_paths = [
    LOCAL_PROD_DIR / "chroma_db",
    LOCAL_PROD_DIR / "bm25_child_chunks.json",
    LOCAL_PROD_DIR / "index_manifest.json",
    LOCAL_PROD_DIR / "parent_docs.json",
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise RuntimeError(
        "Restore không đầy đủ, thiếu:\n"
        + "\n".join(missing_paths)
    )

print("[OK] Restore index thành công.")

Mounted at /content/drive
[INFO] Đang restore full index từ Google Drive...
[OK] Restore index thành công.


In [7]:

import os
import shutil
from pathlib import Path
from google.colab import drive

# ── Cấu hình kiểm thử ────────────────────────────────────────
DEV_MODE = False
DEV_ROWS = 1000
DEV_SAMPLE_SEED = 42

FORCE_REBUILD_INDEX = False
AUTO_RESET_BROKEN_INDEX = True
REQUIRE_GPU = True

# Drive chỉ dùng để backup/restore.
USE_DRIVE_BACKUP = True
RESTORE_FROM_DRIVE = True

storage_name = "dev" if DEV_MODE else "prod"

# Database đang chạy nằm trong ổ local nhanh của Colab.
BASE_DIR = Path(
    f"/content/medical_rag_runtime/storage/{storage_name}"
).resolve()
BASE_DIR.mkdir(parents=True, exist_ok=True)

CHROMA_PATH = str(BASE_DIR / "chroma_db")
PARENT_DOC_PATH = str(BASE_DIR / "parent_docs.json")
BM25_CHUNKS_PATH = str(BASE_DIR / "bm25_child_chunks.json")
INDEX_MANIFEST_PATH = str(BASE_DIR / "index_manifest.json")

# COLLECTION_NAME = (
#     f"medical_hybrid_dev_{DEV_ROWS}_v1"
#     if DEV_MODE
#     else "medical_hybrid_prod_v1"
# )
COLLECTION_NAME = (
    f"medical_hybrid_dev_{DEV_ROWS}_qa_v2"
    if DEV_MODE
    else "medical_hybrid_prod_qa_v2"
)
DRIVE_BACKUP_DIR = Path(
    "/content/drive/MyDrive/medical_rag_backups"
)
BACKUP_NAME = (
    f"medical_rag_dev_{DEV_ROWS}"
    if DEV_MODE
    else "medical_rag_prod"
)
BACKUP_ZIP = DRIVE_BACKUP_DIR / f"{BACKUP_NAME}.zip"

if USE_DRIVE_BACKUP or RESTORE_FROM_DRIVE:
    drive.mount("/content/drive", force_remount=False)
    DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)

if RESTORE_FROM_DRIVE:
    if not BACKUP_ZIP.exists():
        raise FileNotFoundError(
            f"Không tìm thấy backup: {BACKUP_ZIP}"
        )

    if BASE_DIR.exists():
        shutil.rmtree(BASE_DIR, ignore_errors=True)

    BASE_DIR.mkdir(parents=True, exist_ok=True)

    print(f"[INFO] Đang restore {BACKUP_ZIP} về /content...")
    shutil.unpack_archive(str(BACKUP_ZIP), str(BASE_DIR))
    print("[OK] Đã restore index về /content.")

print(f"[OK] Chế độ: {'DEV' if DEV_MODE else 'PROD'}")
print(f"[OK] Số dòng: {DEV_ROWS if DEV_MODE else 'FULL DATASET'}")
print(f"[OK] Runtime storage: {BASE_DIR}")
print(f"[OK] Collection: {COLLECTION_NAME}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[INFO] Đang restore /content/drive/MyDrive/medical_rag_backups/medical_rag_prod.zip về /content...
[OK] Đã restore index về /content.
[OK] Chế độ: PROD
[OK] Số dòng: FULL DATASET
[OK] Runtime storage: /content/medical_rag_runtime/storage/prod
[OK] Collection: medical_hybrid_prod_qa_v2


In [8]:
from pathlib import Path
import shutil


DRIVE_MODEL_ZIP = Path(
    "/content/drive/MyDrive/"
    "medical_rag_models/"
    "bge-m3-model.zip"
)

LOCAL_ZIP = Path(
    "/content/bge-m3-model.zip"
)

LOCAL_MODEL_DIR = Path(
    "/content/models/bge-m3"
)


if not DRIVE_MODEL_ZIP.exists():
    raise FileNotFoundError(
        f"Không tìm thấy:\n{DRIVE_MODEL_ZIP}"
    )

# Copy ZIP từ Drive về ổ local của Colab.
shutil.copy2(
    DRIVE_MODEL_ZIP,
    LOCAL_ZIP,
)

if LOCAL_MODEL_DIR.exists():
    shutil.rmtree(
        LOCAL_MODEL_DIR
    )

LOCAL_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

shutil.unpack_archive(
    str(LOCAL_ZIP),
    str(LOCAL_MODEL_DIR),
)

print(
    "[OK] Đã restore BGE-M3 về:",
    LOCAL_MODEL_DIR,
)

[OK] Đã restore BGE-M3 về: /content/models/bge-m3


In [9]:
from pathlib import Path

model_dir = Path(
    "/content/models/bge-m3"
)

weight_path = (
    model_dir
    / "pytorch_model.bin"
)

if not weight_path.exists():
    raise FileNotFoundError(
        "Thiếu pytorch_model.bin."
    )

if weight_path.stat().st_size < 2_000_000_000:
    raise RuntimeError(
        "pytorch_model.bin chưa đầy đủ."
    )

print("[READY] BGE-M3 local đã đầy đủ.")
print(
    "Weight size:",
    f"{weight_path.stat().st_size / 1024**3:.2f} GiB",
)

[READY] BGE-M3 local đã đầy đủ.
Weight size: 2.12 GiB


Mới

In [10]:
import gc
import os
import shutil
from pathlib import Path

import chromadb
import torch
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi


# =========================================================
# 1. GPU
# =========================================================

if torch.cuda.is_available():
    DEVICE = "cuda"
    GPU_NAME = torch.cuda.get_device_name(0)

    print(
        f"[OK] GPU CUDA đã sẵn sàng: "
        f"{GPU_NAME}"
    )
else:
    DEVICE = "cpu"
    GPU_NAME = None

    if REQUIRE_GPU:
        raise RuntimeError(
            "Không phát hiện GPU CUDA trên Colab. "
            "Vào Runtime → Change runtime type → T4 GPU, "
            "sau đó Restart session và chạy lại."
        )

    print("[WARN] Không có CUDA; đang chạy bằng CPU.")


ENCODE_BATCH_SIZE = 128
CHROMA_BATCH_SIZE = 512


# =========================================================
# 2. Kiểm tra đường dẫn model local
# =========================================================

if "BGE_LOCAL_MODEL_PATH" not in globals():
    BASE_MODEL_DIR = Path(
        "/content/models/bge-m3"
    )

    MODEL_CANDIDATES = [
        BASE_MODEL_DIR,
        BASE_MODEL_DIR / "bge-m3",
    ]

    BGE_LOCAL_MODEL_PATH = None

    for candidate in MODEL_CANDIDATES:
        if (
            (candidate / "config.json").exists()
            and (candidate / "modules.json").exists()
            and (
                candidate
                / "pytorch_model.bin"
            ).exists()
        ):
            BGE_LOCAL_MODEL_PATH = candidate
            break


if BGE_LOCAL_MODEL_PATH is None:
    raise FileNotFoundError(
        "Không tìm thấy BGE-M3 local. "
        "Hãy chạy cell restore model trước."
    )


weight_path = (
    BGE_LOCAL_MODEL_PATH
    / "pytorch_model.bin"
)

if not weight_path.exists():
    raise FileNotFoundError(
        f"Thiếu file trọng số:\n{weight_path}"
    )

if weight_path.stat().st_size < 2_000_000_000:
    raise RuntimeError(
        "pytorch_model.bin chưa đầy đủ."
    )

print(
    "[OK] Sử dụng BGE-M3 local:",
    BGE_LOCAL_MODEL_PATH,
)


# =========================================================
# 3. Embedding function
# =========================================================

class BGEEmbeddingFunction:
    def __init__(
        self,
        model_name=BGE_LOCAL_MODEL_PATH,
        device=DEVICE,
        encode_batch_size=ENCODE_BATCH_SIZE,
    ):
        self.device = device
        self.encode_batch_size = (
            encode_batch_size
        )

        model_path = Path(model_name)

        if not model_path.exists():
            raise FileNotFoundError(
                "Không tìm thấy model local tại:\n"
                f"{model_path}"
            )

        print(
            "[INFO] Đang load BGE-M3 từ local:",
            model_path,
        )

        self.model = SentenceTransformer(
            str(model_path),
            device=device,

            # Quan trọng:
            # không được tải thêm từ Hugging Face.
            local_files_only=True,
        )

        print(
            f"[OK] BGE-M3 device="
            f"{self.model.device} | "
            f"encode_batch_size="
            f"{self.encode_batch_size}"
        )

    def __call__(self, input):
        input_texts = (
            [input]
            if isinstance(input, str)
            else list(input)
        )

        embeddings = self.model.encode(
            input_texts,
            batch_size=self.encode_batch_size,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
        )

        return embeddings.tolist()

    def name(self):
        return (
            "bge-m3-normalized-"
            "cosine-colab-gpu"
        )

    def embed_query(self, input):
        return self.__call__(input)

    def embed_documents(self, input):
        return self.__call__(input)


# =========================================================
# 4. Mở ChromaDB
# =========================================================

def open_local_chroma():
    global chroma_client, collection

    Path(CHROMA_PATH).mkdir(
        parents=True,
        exist_ok=True,
    )

    try:
        chroma_client = (
            chromadb.PersistentClient(
                path=CHROMA_PATH
            )
        )

        collection = (
            chroma_client
            .get_or_create_collection(
                name=COLLECTION_NAME,
                embedding_function=embedding_fn,
                metadata={
                    "hnsw:space": "cosine"
                },
            )
        )

        return collection.count()

    except Exception as error:
        if not AUTO_RESET_BROKEN_INDEX:
            raise RuntimeError(
                "Không mở được ChromaDB local.\n"
                f"Path: {CHROMA_PATH}\n"
                f"Chi tiết: {error}"
            ) from error

        print(
            "[WARN] ChromaDB bị lỗi:",
            error,
        )

        print(
            "[INFO] Đang xóa Chroma local "
            "bị hỏng và tạo lại..."
        )

        for variable_name in [
            "collection",
            "chroma_client",
        ]:
            if variable_name in globals():
                del globals()[variable_name]

        gc.collect()

        shutil.rmtree(
            CHROMA_PATH,
            ignore_errors=True,
        )

        Path(CHROMA_PATH).mkdir(
            parents=True,
            exist_ok=True,
        )

        if os.path.exists(
            INDEX_MANIFEST_PATH
        ):
            os.remove(
                INDEX_MANIFEST_PATH
            )

        chroma_client = (
            chromadb.PersistentClient(
                path=CHROMA_PATH
            )
        )

        collection = (
            chroma_client
            .get_or_create_collection(
                name=COLLECTION_NAME,
                embedding_function=embedding_fn,
                metadata={
                    "hnsw:space": "cosine"
                },
            )
        )

        return 0


# =========================================================
# 5. Khởi tạo model local và mở Chroma
# =========================================================

embedding_fn = BGEEmbeddingFunction(
    model_name=BGE_LOCAL_MODEL_PATH,
)

current_count = open_local_chroma()

print(
    f"[INFO] ChromaDB hiện có: "
    f"{current_count} chunks"
)

print(
    f"[INFO] Embedding device: "
    f"{embedding_fn.model.device}"
)

[OK] GPU CUDA đã sẵn sàng: NVIDIA L4
[OK] Sử dụng BGE-M3 local: /content/models/bge-m3
[INFO] Đang load BGE-M3 từ local: /content/models/bge-m3


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[OK] BGE-M3 device=cuda:0 | encode_batch_size=128
[INFO] ChromaDB hiện có: 413344 chunks
[INFO] Embedding device: cuda:0


In [11]:
def chunk_text_by_words(text, chunk_size=250, overlap=50):
    words = str(text).split()
    if len(words) <= chunk_size:
        return [str(text)]

    chunks = []
    step = chunk_size - overlap
    for i in range(0, len(words), step):
        chunks.append(" ".join(words[i:i + chunk_size]))
        if i + chunk_size >= len(words):
            break
    return chunks


def atomic_json_dump(data, path):
    """Ghi file tạm rồi đổi tên để tránh file JSON bị dở dang."""
    temp_path = f"{path}.tmp"
    with open(temp_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)
    os.replace(temp_path, path)


def load_manifest():
    if not os.path.exists(INDEX_MANIFEST_PATH):
        return None
    try:
        with open(INDEX_MANIFEST_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        print(f"[WARN] Không đọc được index manifest: {e}")
        return None


def index_is_ready():
    manifest = load_manifest()
    if not manifest:
        return False

    required_files_exist = (
        os.path.exists(PARENT_DOC_PATH)
        and os.path.exists(BM25_CHUNKS_PATH)
    )

    return (
        manifest.get("status") == "complete"
        and manifest.get("collection_name") == COLLECTION_NAME
        and bool(manifest.get("dev_mode")) == DEV_MODE
        and int(manifest.get("dev_rows", -1)) == (
            DEV_ROWS if DEV_MODE else -1
        )
        and required_files_exist
        and int(manifest.get("expected_chunks", -1)) == collection.count()
        and collection.count() > 0
    )


def create_medical_collection():
    return chroma_client.get_or_create_collection(
        name=COLLECTION_NAME,
        embedding_function=embedding_fn,
        metadata={"hnsw:space": "cosine"},
    )


def recreate_collection():
    global collection

    try:
        chroma_client.delete_collection(COLLECTION_NAME)
        print(
            f"[INFO] Đã xóa collection chưa hoàn chỉnh: "
            f"{COLLECTION_NAME}"
        )
    except Exception:
        pass

    collection = create_medical_collection()


child_chunks = []
parent_documents = {}

if FORCE_REBUILD_INDEX:
    recreate_collection()
    if os.path.exists(INDEX_MANIFEST_PATH):
        os.remove(INDEX_MANIFEST_PATH)

ready = index_is_ready()

# Thử load index hoàn chỉnh.
if ready:
    try:
        with open(PARENT_DOC_PATH, "r", encoding="utf-8") as f:
            parent_documents = {
                int(k): v for k, v in json.load(f).items()
            }

        with open(BM25_CHUNKS_PATH, "r", encoding="utf-8") as f:
            child_chunks = json.load(f)

        manifest = load_manifest()
        expected_chunks = int(manifest["expected_chunks"])

        if len(child_chunks) != expected_chunks:
            raise RuntimeError(
                f"BM25 chunks={len(child_chunks)} nhưng manifest={expected_chunks}"
            )

        print(
            f"[OK] Loaded {len(parent_documents)} parents, "
            f"{len(child_chunks)} child chunks."
        )
    except Exception as e:
        print(f"[WARN] Index local không hợp lệ: {e}")
        ready = False

# Nếu index thiếu, hỏng hoặc bị ngắt giữa chừng thì xây lại sạch.
if not ready:
    if collection.count() > 0:
        recreate_collection()

    if os.path.exists(INDEX_MANIFEST_PATH):
        os.remove(INDEX_MANIFEST_PATH)

    print("[...] 🚀 Đang xây dựng lại toàn bộ index...")
    print("[...] Đang tải dữ liệu từ Hugging Face...")

    # Chỉ đọc đúng hai cột đã làm sạch, không tải question/answer gốc.
    df_raw = pd.read_parquet(
        "hf://datasets/ynguyen1010/medical_vietnamese_datasets/"
        "cleaned_format/train-00000-of-00001.parquet",
        columns=["question_cleaned", "answer_cleaned"],
    )

    # Loại bỏ dữ liệu thiếu hoặc chuỗi rỗng.
    df_raw = (
        df_raw
        .dropna(subset=["question_cleaned", "answer_cleaned"])
        .copy()
    )

    df_raw["question_cleaned"] = (
        df_raw["question_cleaned"].astype(str).str.strip()
    )
    df_raw["answer_cleaned"] = (
        df_raw["answer_cleaned"].astype(str).str.strip()
    )

    df_raw = df_raw[
        (df_raw["question_cleaned"] != "")
        & (df_raw["answer_cleaned"] != "")
    ].reset_index(drop=True)

    print(
        "[OK] Chỉ sử dụng hai cột cleaned: "
        f"{list(df_raw.columns)}"
    )
    print(f"[OK] Dữ liệu hợp lệ sau lọc: {df_raw.shape}")

    if DEV_MODE:
        sample_size = min(DEV_ROWS, len(df_raw))

        df_raw = (
            df_raw
            .sample(
                n=sample_size,
                random_state=DEV_SAMPLE_SEED,
            )
            .reset_index(drop=True)
        )

        print(
            f"[DEV MODE] Chỉ index thử {len(df_raw)} dòng "
            f"(seed={DEV_SAMPLE_SEED})."
        )
    else:
        print("[PROD MODE] Sử dụng toàn bộ dataset.")

    for parent_id, (_, row) in enumerate(
        tqdm(
            df_raw.iterrows(),
            total=len(df_raw),
            desc="Tạo Parent-Child Chunks"
        )
    ):
        # Dùng nghiêm ngặt dữ liệu đã làm sạch, không fallback cột gốc.
        q = row["question_cleaned"]
        a = row["answer_cleaned"]

        parent_documents[parent_id] = {
            "question": q,
            "answer": a
        }

        chunks = chunk_text_by_words(a)

        for chunk_idx, chunk in enumerate(chunks):
            retrieval_text = (
                f"Câu hỏi: {q}\n"
                f"Nội dung trả lời: {chunk}"
            )

            child_chunks.append({
                "id": f"doc_{parent_id}_chunk_{chunk_idx}",
                "parent_id": parent_id,

                # Answer chunk sạch dùng làm evidence cho CRAG và generator.
                "text": chunk,

                # Lưu câu hỏi để metadata và reranking sử dụng.
                "question": q,

                # Nội dung dùng để tạo embedding và BM25 index.
                "retrieval_text": retrieval_text
            })

    expected_chunks = len(child_chunks)
    print(f"[INFO] Tổng child chunks: {expected_chunks}")

    # Lưu dữ liệu phụ trước; manifest chỉ được tạo sau khi embedding hoàn tất.
    atomic_json_dump(
        {str(k): v for k, v in parent_documents.items()},
        PARENT_DOC_PATH
    )
    atomic_json_dump(child_chunks, BM25_CHUNKS_PATH)

    batch_size = CHROMA_BATCH_SIZE

    for i in tqdm(
        range(0, expected_chunks, batch_size),
        desc=f"Embedding vào ChromaDB local ({DEVICE})"
    ):
        batch = child_chunks[i:i + batch_size]
        # Nội dung dùng để tạo vector:
        # question_cleaned + answer_cleaned chunk
        retrieval_texts = [
            c["retrieval_text"]
            for c in batch
        ]

        # Tự tạo embedding bằng BGE-M3.
        batch_embeddings = embedding_fn(
            retrieval_texts
        )

        collection.add(
            ids=[
                c["id"]
                for c in batch
            ],

            # ChromaDB trả về answer chunk sạch.
            documents=[
                c["text"]
                for c in batch
            ],

            # Vector được tạo từ question + answer chunk.
            embeddings=batch_embeddings,

            metadatas=[
                {
                    "parent_id": c["parent_id"],
                    "question": c["question"]
                }
                for c in batch
            ]
        )

    actual_chunks = collection.count()
    if actual_chunks != expected_chunks:
        raise RuntimeError(
            f"Index chưa hoàn chỉnh: expected={expected_chunks}, "
            f"actual={actual_chunks}"
        )

    manifest = {
        "status": "complete",
        "collection_name": COLLECTION_NAME,
        "dev_mode": DEV_MODE,
        "dev_rows": DEV_ROWS if DEV_MODE else -1,
        "dev_sample_seed": DEV_SAMPLE_SEED if DEV_MODE else None,
        "dataset_rows": len(df_raw),
        "expected_chunks": expected_chunks,
        "actual_chunks": actual_chunks,
        "chunk_size": 250,
        "overlap": 50,
        "embedding_model": "BAAI/bge-m3",
        "embedding_device": str(embedding_fn.model.device),
        "encode_batch_size": ENCODE_BATCH_SIZE,
        "chroma_batch_size": CHROMA_BATCH_SIZE,
        "source_columns": [
            "question_cleaned",
            "answer_cleaned"
        ],
        "embedded_field": ("question_cleaned_plus_answer_cleaned_chunk"),
        "embedding_strategy": ("question_aware_child_embedding_v2"),
        "distance_metric": "cosine",
        "created_at": time.time()
    }
    atomic_json_dump(manifest, INDEX_MANIFEST_PATH)

    print(
        f"\n[OK] ✅ HOÀN THÀNH INDEX: {actual_chunks} chunks "
        f"đã lưu tại {BASE_DIR}."
    )

# Giữ nguyên cách token hóa BM25 theo yêu cầu của bạn.
#tokenized_corpus = [c["text"].lower().split() for c in child_chunks]
tokenized_corpus = [
    c["retrieval_text"].lower().split()
    for c in child_chunks
]
bm25 = BM25Okapi(tokenized_corpus)
print("[OK] Hoàn thành khởi tạo Vector Store & BM25 Sparse Retriever.")


def backup_completed_index():
    """Nén index hoàn chỉnh trong /content rồi copy lên Drive."""
    if not USE_DRIVE_BACKUP:
        return

    if not index_is_ready():
        print("[WARN] Index chưa hoàn chỉnh nên chưa backup.")
        return

    DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)

    temp_base = Path("/content") / BACKUP_NAME
    temp_zip = Path(
        shutil.make_archive(
            str(temp_base),
            "zip",
            root_dir=str(BASE_DIR),
        )
    )

    shutil.copy2(temp_zip, BACKUP_ZIP)
    temp_zip.unlink(missing_ok=True)

    print(f"[OK] Đã backup index lên: {BACKUP_ZIP}")


#backup_completed_index()


[OK] Loaded 68498 parents, 413344 child chunks.
[OK] Hoàn thành khởi tạo Vector Store & BM25 Sparse Retriever.


## 4. Cross-Encoder Reranker

Sử dụng mô hình Reranker đa ngôn ngữ để sắp xếp lại (Rerank) văn bản sau khi kết hợp kết quả từ Dense và Sparse Search.

In [12]:
from sentence_transformers import CrossEncoder

print(
    f"[...] Đang tải Cross-Encoder Reranker "
    f"trên {DEVICE}..."
)

reranker = CrossEncoder(
    "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
    device=DEVICE,
)

print(
    f"[OK] Reranker đã tải xong trên {DEVICE}"
    + (f": {GPU_NAME}" if GPU_NAME else "")
)


[...] Đang tải Cross-Encoder Reranker trên cuda...


config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

[OK] Reranker đã tải xong trên cuda: NVIDIA L4


**Nếu download model chậm thì có thể load từ drive đã tải trước**

In [ ]:
from pathlib import Path
import shutil
import zipfile


DRIVE_RERANKER_ZIP = Path(
    "/content/drive/MyDrive/"
    "medical_rag_models/"
    "mmarco-reranker.zip"
)

LOCAL_RERANKER_ZIP = Path(
    "/content/mmarco-reranker.zip"
)

LOCAL_RERANKER_DIR = Path(
    "/content/models/mmarco-reranker"
)


def reranker_is_ready() -> bool:
    required_files = [
        LOCAL_RERANKER_DIR / "config.json",
        LOCAL_RERANKER_DIR / "model.safetensors",
    ]

    return all(
        path.exists()
        for path in required_files
    )


if reranker_is_ready():
    print(
        "[OK] Reranker đã có sẵn trong /content."
    )

else:
    if not DRIVE_RERANKER_ZIP.exists():
        raise FileNotFoundError(
            "Không tìm thấy reranker backup:\n"
            f"{DRIVE_RERANKER_ZIP}"
        )

    if not zipfile.is_zipfile(
        DRIVE_RERANKER_ZIP
    ):
        raise RuntimeError(
            "mmarco-reranker.zip "
            "không phải ZIP hợp lệ."
        )

    if LOCAL_RERANKER_ZIP.exists():
        LOCAL_RERANKER_ZIP.unlink()

    print(
        "[INFO] Đang copy reranker từ Drive..."
    )

    shutil.copy2(
        DRIVE_RERANKER_ZIP,
        LOCAL_RERANKER_ZIP,
    )

    if LOCAL_RERANKER_DIR.exists():
        shutil.rmtree(
            LOCAL_RERANKER_DIR
        )

    LOCAL_RERANKER_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    print("[INFO] Đang giải nén reranker...")

    shutil.unpack_archive(
        str(LOCAL_RERANKER_ZIP),
        str(LOCAL_RERANKER_DIR),
    )

    if not reranker_is_ready():
        raise RuntimeError(
            "Đã giải nén nhưng reranker "
            "vẫn thiếu file."
        )

    print(
        "[OK] Đã restore reranker về:",
        LOCAL_RERANKER_DIR,
    )

[INFO] Đang copy reranker từ Drive...
[INFO] Đang giải nén reranker...
[OK] Đã restore reranker về: /content/models/mmarco-reranker


In [ ]:
from pathlib import Path
from sentence_transformers import CrossEncoder


RERANKER_LOCAL_PATH = Path(
    "/content/models/mmarco-reranker"
)

required_files = [
    RERANKER_LOCAL_PATH / "config.json",
    RERANKER_LOCAL_PATH / "model.safetensors",
]

missing_files = [
    path
    for path in required_files
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Reranker local còn thiếu:\n"
        + "\n".join(
            str(path)
            for path in missing_files
        )
    )


print(
    "[INFO] Đang load Cross-Encoder "
    "từ local:",
    RERANKER_LOCAL_PATH,
)

reranker = CrossEncoder(
    str(RERANKER_LOCAL_PATH),
    device=DEVICE,

    # Không cho phép quay lại Hugging Face.
    local_files_only=True,
)

print(
    f"[OK] Reranker đã load xong trên {DEVICE}"
    + (
        f": {GPU_NAME}"
        if GPU_NAME
        else ""
    )
)

[INFO] Đang load Cross-Encoder từ local: /content/models/mmarco-reranker


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[OK] Reranker đã load xong trên cuda: NVIDIA L4


## 5. Xây dựng Kiến Trúc Pipeline (Có Cache & Memory)

Các module ánh xạ 1-1 với sơ đồ cấu trúc, tích hợp Cache + Conversation Memory.

In [17]:
# ═══════════════════════════════════════════════════════════════
# MODULE 1: Query Processing (Spelling Fix + HyDE) — CÓ CACHE
# ═══════════════════════════════════════════════════════════════

def query_processing(user_query: str) -> dict:
    user_query = str(user_query).strip()

    # Tầng 1 Cache: Spell Correction
    fixed_query = cache.get(cache.spell_cache, user_query)
    if fixed_query:
        print("      > [CACHE HIT ⚡] Spell correction")
    else:
        spell_sys = """
Bạn là bộ sửa lỗi chính tả tiếng Việt.
Nhiệm vụ duy nhất:
Sửa lỗi gõ phím, lỗi dấu, lỗi chính tả và viết tắt trong câu đầu vào.
Quy tắc bắt buộc:
1. Không trả lời câu hỏi.
2. Không bổ sung thông tin.
3. Không mở rộng ý nghĩa.
4. Không thay đổi chủ thể hoặc ý định.
5. Giữ nguyên dạng câu hỏi nếu đầu vào là câu hỏi.
6. Chỉ trả về đúng câu sau khi sửa.
7. Nếu câu đã đúng, trả lại nguyên văn.

Không thêm giải thích hoặc Markdown.
""".strip()
        llm_fixed_query = call_llm(
            user_query,
            system_instruction=spell_sys,
            temperature=0.0,
            task_name="spell_correction"
        )

        if is_valid_llm_output(llm_fixed_query):
            fixed_query = llm_fixed_query.strip()
            cache.set(cache.spell_cache, user_query, fixed_query)
        else:
            # LLM lỗi thì dùng câu gốc, không cache lỗi để lần sau còn thử lại.
            fixed_query = user_query

    # Tầng 2 Cache: HyDE
    hyde_doc = cache.get(cache.hyde_cache, fixed_query)
    if hyde_doc:
        print("      > [CACHE HIT ⚡] HyDE document")
    else:
        hyde_sys = (
            "Bạn là bác sĩ. Hãy viết một đoạn văn ngắn (50 từ) mô phỏng "
            "câu trả lời trực tiếp cho câu hỏi y tế của người bệnh. "
            "Không format phức tạp."
        )
        llm_hyde_doc = call_llm(
            fixed_query,
            system_instruction=hyde_sys,
            temperature=0.5,
            task_name="hyde_generation"
        )

        if is_valid_llm_output(llm_hyde_doc):
            hyde_doc = llm_hyde_doc.strip()
            cache.set(cache.hyde_cache, fixed_query, hyde_doc)
        else:
            # Nếu HyDE lỗi, dense retrieval vẫn chạy bằng query đã sửa.
            hyde_doc = ""

    # Giữ nguyên cách ghép Query + HyDE theo yêu cầu của bạn.
    search_str = f"{fixed_query} {hyde_doc}".strip()

    return {
        "original": user_query,
        "fixed": fixed_query,
        "hyde": hyde_doc,
        "search_str": search_str
    }


# ═══════════════════════════════════════════════════════════════
# MODULE 2: Hybrid Search & Reranking
# Rerank CHILD trước, sau đó mới lấy PARENT đầy đủ.
# ═══════════════════════════════════════════════════════════════

def hybrid_retrieve_and_rerank(q_dict: dict, top_k: int = 3) -> list:
    candidate_k = max(top_k * 2, top_k)

    # 1. Dense Search
    vector_res = collection.query(
        query_texts=[q_dict["search_str"]],
        n_results=candidate_k
    )

    dense_ids = vector_res["ids"][0]
    dense_docs = vector_res["documents"][0]
    dense_metas = vector_res["metadatas"][0]

    # 2. BM25 Sparse Search
    # Giữ nguyên cách tách từ bằng split() theo yêu cầu của bạn.
    tokenized_query = q_dict["fixed"].lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)

    # Chỉ lấy tài liệu BM25 có score > 0 để tránh đưa tài liệu rác vào reranker.
    sorted_bm25_indices = np.argsort(bm25_scores)[::-1]
    top_bm25_idx = [
        int(i)
        for i in sorted_bm25_indices
        if bm25_scores[i] > 0
    ][:candidate_k]

    # 3. Gộp các CHILD ứng viên và khử trùng lặp theo child_id
    candidate_children = {}

    for child_id, doc, meta in zip(dense_ids,dense_docs,dense_metas):
        question = str(meta.get("question", "")).strip()

        candidate_children[str(child_id)] = {
            "child_id": str(child_id),

            # Answer chunk sạch dùng làm evidence.
            "text": doc,

            # Cross-Encoder sẽ chấm question + answer chunk.
            "rerank_text": (
                f"Câu hỏi: {question}\n"
                f"Nội dung trả lời: {doc}"
            ),

            "parent_id": int(meta["parent_id"]),
            "source": "dense"
        }

    for i in top_bm25_idx:
      child = child_chunks[i]
      child_id = str(child["id"])

      if child_id not in candidate_children:
          candidate_children[child_id] = {
              "child_id": child_id,

              # Evidence answer chunk sạch.
              "text": child["text"],

              # Question + answer chunk dùng cho reranking.
              "rerank_text": child[
                  "retrieval_text"
              ],

              "parent_id": int(
                  child["parent_id"]
              ),
              "source": "bm25"
          }

    candidates = list(candidate_children.values())
    if not candidates:
        return []

    # 4. Cross-Encoder rerank trên CHILD ngắn để tránh parent dài bị truncate.
    pairs = [
        [
            q_dict["fixed"],
            candidate.get(
                "rerank_text",
                candidate["text"]
            )
        ]
        for candidate in candidates
    ]
    scores = np.asarray(reranker.predict(pairs), dtype=float).reshape(-1)
    ranked_child_indices = np.argsort(scores)[::-1]

    # 5. Chọn top_k PARENT khác nhau dựa trên child tốt nhất của mỗi parent.
    final_docs = []
    selected_parent_ids = set()

    for idx in ranked_child_indices:
        candidate = candidates[int(idx)]
        parent_id = int(candidate["parent_id"])

        if parent_id in selected_parent_ids:
            continue

        parent = parent_documents.get(parent_id)
        if not parent:
            continue

        selected_parent_ids.add(parent_id)
        final_docs.append({
            # Parent đầy đủ dùng cho generator.
            "doc": (
                f"Hỏi: {parent['question']}\n"
                f"Đáp: {parent['answer']}"
            ),

            "id": parent_id,

            # Lưu riêng câu hỏi của tài liệu để CRAG kiểm tra chủ đề.
            "question": parent["question"],

            "score": float(
                scores[int(idx)]
            ),

            # Child chunk tốt nhất được Cross-Encoder chọn.
            "evidence_chunk": candidate["text"],

            "retrieval_source": candidate["source"]
        })

        if len(final_docs) >= top_k:
            break

    return final_docs


# ═══════════════════════════════════════════════════════════════
# MODULE 3: CRAG Grader & Knowledge Refinement
# ═══════════════════════════════════════════════════════════════

def parse_json_object(text):
    """
    Parse JSON kể cả khi LLM:
    - bọc bằng ```json ... ```;
    - thêm khoảng trắng;
    - thêm vài ký tự trước hoặc sau JSON.
    """
    if not is_valid_llm_output(text):
        return None

    cleaned = str(text).strip()

    # Bỏ Markdown code fence.
    cleaned = re.sub(
        r"^```(?:json)?\s*",
        "",
        cleaned,
        flags=re.IGNORECASE
    )

    cleaned = re.sub(
        r"\s*```$",
        "",
        cleaned
    )

    # Thử parse trực tiếp.
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    # Tìm JSON object từ dấu { đầu tiên đến dấu } cuối cùng.
    start = cleaned.find("{")
    end = cleaned.rfind("}")

    if start == -1 or end == -1 or end <= start:
        return None

    try:
        return json.loads(
            cleaned[start:end + 1]
        )
    except json.JSONDecodeError:
        return None



ENABLE_TAVILY_FALLBACK = True


CRAG_SEPARATOR = "|||"

STRICT_CRAG_SYSTEM_PROMPT = """
Bạn là bộ kiểm định bằng chứng nghiêm ngặt cho hệ thống RAG y tế.

Nhiệm vụ của bạn là đánh giá ĐỘC LẬP từng tài liệu được cung cấp.
Không sử dụng kiến thức bên ngoài tài liệu.

Mỗi tài liệu phải được đánh giá theo đúng ba điều kiện:

1. same_subject
2. answers_question
3. direct_evidence


══════════════════════════════════════════════════════════════
1. SAME_SUBJECT — CÓ ĐÚNG ĐỐI TƯỢNG ĐƯỢC HỎI KHÔNG?
══════════════════════════════════════════════════════════════

same_subject=true chỉ khi câu hỏi người dùng và tài liệu nói về
chính xác cùng một:

- bệnh;
- tình trạng;
- triệu chứng;
- thuốc;
- phương pháp điều trị;
- biến chủng;
- tác nhân;
- hoặc đối tượng y tế.

Có thể xem là cùng chủ đề khi hai bên dùng tên gọi khác nhau nhưng
rõ ràng là tên đồng nghĩa của cùng một đối tượng.

same_subject=false khi xảy ra bất kỳ trường hợp nào sau đây:

- Chỉ trùng một vài từ khóa.
- Chỉ cùng bộ phận cơ thể.
- Chỉ cùng nói về mắt, tim, phổi, xương khớp hoặc giấc ngủ.
- Chỉ cùng nhóm bệnh chung.
- Chỉ có một triệu chứng giống nhau.
- Tài liệu nói về một bệnh, biến chủng hoặc tình trạng khác.
- Phải suy luận hoặc sử dụng kiến thức bên ngoài mới cho rằng liên quan.

Ví dụ:

Câu hỏi người dùng:
"Triệu chứng của biến chủng Covid JN.1 là gì?"

Tài liệu:
"Triệu chứng của viêm khớp dạng thấp"

Kết luận bắt buộc:
same_subject=false

Lý do:
Covid JN.1 và viêm khớp dạng thấp là hai đối tượng khác nhau,
dù cả hai tài liệu đều có thể chứa từ "triệu chứng".


══════════════════════════════════════════════════════════════
2. ANSWERS_QUESTION — CÓ ĐÚNG Ý ĐỊNH CÂU HỎI KHÔNG?
══════════════════════════════════════════════════════════════

Trước tiên, xác định người dùng đang hỏi loại thông tin nào.

Nếu câu hỏi yêu cầu cụ thể:

- hỏi khái niệm:
  Evidence phải giải thích khái niệm hoặc định nghĩa.

- hỏi nguyên nhân:
  Evidence phải trực tiếp nêu nguyên nhân hoặc yếu tố nguy cơ.

- hỏi triệu chứng:
  Evidence phải trực tiếp nêu triệu chứng hoặc biểu hiện.

- hỏi tác hại:
  Evidence phải trực tiếp nêu tác hại, ảnh hưởng hoặc hậu quả.

- hỏi biến chứng:
  Evidence phải trực tiếp nêu biến chứng.

- hỏi điều trị:
  Evidence phải trực tiếp nêu phương pháp điều trị.

- hỏi thuốc:
  Evidence phải trực tiếp nêu thuốc hoặc nhóm thuốc.

- hỏi tác dụng phụ:
  Evidence phải trực tiếp nêu tác dụng phụ của đúng thuốc
  hoặc phương pháp được hỏi.

- hỏi phòng ngừa:
  Evidence phải trực tiếp nêu biện pháp phòng ngừa.

- hỏi thông tin mới nhất, hiện tại hoặc trong một năm cụ thể:
  Evidence phải trực tiếp nói về đúng thời điểm hoặc phiên bản được hỏi.
  Tài liệu chung chung hoặc không có thông tin thời gian là chưa đủ.

Nếu người dùng chỉ nhập tên một bệnh, tình trạng hoặc hiện tượng
mà không nêu ý định cụ thể, hãy xem đó là yêu cầu thông tin tổng quan.

Ví dụ:

"Hiện tượng ngủ mở mắt"

được xem là câu hỏi tổng quan.

Với câu hỏi tổng quan, answers_question=true khi Evidence trực tiếp
cung cấp ít nhất một thông tin hữu ích về đúng chủ đề, chẳng hạn:

- khái niệm;
- đặc điểm;
- nguyên nhân;
- triệu chứng;
- ảnh hưởng;
- biến chứng;
- chẩn đoán;
- điều trị;
- phòng ngừa.

answers_question=false khi:

- Tài liệu đúng chủ đề nhưng không trả lời loại thông tin được hỏi.
- Người dùng hỏi triệu chứng nhưng Evidence chỉ nói nguyên nhân.
- Người dùng hỏi tác hại nhưng Evidence chỉ nói điều trị.
- Người dùng hỏi thông tin mới nhất nhưng Evidence không có thông tin
  về thời điểm hoặc phiên bản tương ứng.
- Cần suy luận thêm mới tạo được câu trả lời.


══════════════════════════════════════════════════════════════
3. DIRECT_EVIDENCE — EVIDENCE CÓ THẬT SỰ CHỨA CÂU TRẢ LỜI KHÔNG?
══════════════════════════════════════════════════════════════

direct_evidence=true chỉ khi đoạn Evidence trực tiếp chứa thông tin
có thể dùng để trả lời câu hỏi người dùng.

Chỉ được kiểm tra nội dung bên trong phần:

Evidence:

Không được xem các phần sau là bằng chứng trả lời:

- ID tài liệu;
- câu hỏi của tài liệu;
- tiêu đề tài liệu;
- tên tài liệu;
- kiến thức có sẵn của bạn.

Câu hỏi hoặc tiêu đề tài liệu chỉ được dùng để xác định chủ đề.

direct_evidence=false khi:

- Evidence nói về nội dung khác.
- Evidence chỉ có từ khóa giống câu hỏi.
- Evidence quá mơ hồ.
- Evidence không chứa câu trả lời trực tiếp.
- Cần dùng kiến thức bên ngoài để bổ sung.
- Chỉ có tiêu đề phù hợp nhưng Evidence không phù hợp.
- Không thể tìm được một đoạn trích nguyên văn hỗ trợ câu trả lời.


══════════════════════════════════════════════════════════════
4. QUY TẮC EVIDENCE_QUOTE
══════════════════════════════════════════════════════════════

Khi direct_evidence=true:

- evidence_quote bắt buộc phải có nội dung.
- evidence_quote phải gồm từ 8 đến 30 từ liên tiếp.
- Phải sao chép NGUYÊN VĂN từ phần Evidence của đúng tài liệu.
- Phải giữ nguyên từ ngữ, thứ tự từ và nội dung.
- Không được tóm tắt.
- Không được diễn đạt lại.
- Không được thay từ đồng nghĩa.
- Không được thêm từ.
- Không được bỏ từ ở giữa.
- Không được lấy nội dung từ tiêu đề hoặc câu hỏi tài liệu.
- Không đặt evidence_quote trong dấu ngoặc kép.
- evidence_quote phải nằm trên cùng một dòng.
- evidence_quote không được chứa chuỗi phân cách |||.

Trước khi trả kết quả, hãy tự kiểm tra:

evidence_quote có xuất hiện nguyên văn và liên tiếp
trong phần Evidence hay không?

Nếu câu trả lời là không:

- đặt direct_evidence=false;
- để evidence_quote trống.

Khi direct_evidence=false:

- evidence_quote bắt buộc phải là chuỗi trống;
- không được tạo quote gần đúng;
- không được paraphrase Evidence.


══════════════════════════════════════════════════════════════
5. ĐIỀU KIỆN CHẤP NHẬN
══════════════════════════════════════════════════════════════

Một tài liệu chỉ đủ điều kiện khi đồng thời:

same_subject=true
AND answers_question=true
AND direct_evidence=true
AND evidence_quote là đoạn trích nguyên văn hợp lệ.

Nếu bất kỳ điều kiện nào không chắc chắn, hãy đặt điều kiện đó là false.

Ưu tiên loại nhầm một tài liệu còn hơn chấp nhận một tài liệu không
trực tiếp hỗ trợ câu trả lời.


══════════════════════════════════════════════════════════════
6. ĐỊNH DẠNG OUTPUT
══════════════════════════════════════════════════════════════

Trả về đúng một dòng cho mỗi tài liệu theo định dạng:

document_index|||same_subject|||answers_question|||direct_evidence|||evidence_quote|||reason

Ví dụ tài liệu hợp lệ:

0|||true|||true|||true|||Đoạn trích nguyên văn liên tiếp được sao chép từ Evidence|||Evidence trực tiếp trả lời đúng chủ đề và ý định.

Ví dụ tài liệu không hợp lệ:

1|||false|||false|||false||||||Tài liệu nói về một bệnh hoặc tình trạng khác.

Quy tắc output bắt buộc:

- Không trả JSON.
- Không thêm Markdown.
- Không viết tiêu đề.
- Không thêm lời giải thích trước hoặc sau kết quả.
- Có đúng một dòng cho mỗi tài liệu đầu vào.
- Giữ nguyên document_index.
- Sắp xếp kết quả theo document_index tăng dần.
- Boolean chỉ được viết là true hoặc false.
- Mỗi kết quả chỉ nằm trên một dòng.
- Không xuống dòng bên trong evidence_quote.
- Không xuống dòng bên trong reason.
- evidence_quote không được chứa |||.
- reason không được chứa |||.
- reason phải ngắn gọn, tối đa 25 từ.
- Không được bỏ sót bất kỳ tài liệu nào.
""".strip()


def parse_crag_boolean(value: str) -> bool:
    return str(value).strip().lower() == "true"


def normalize_crag_text(text: str) -> str:
    text = unicodedata.normalize(
        "NFC",
        str(text)
    ).lower()

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


def crag_quote_is_valid(
    quote: str,
    evidence: str
) -> bool:
    normalized_quote = normalize_crag_text(
        quote
    )

    normalized_evidence = normalize_crag_text(
        evidence
    )

    return bool(
        normalized_quote
        and normalized_quote in normalized_evidence
    )


def parse_crag_lines(
    response: str,
    number_of_documents: int
) -> dict:
    """
    Parse output:

    index|||same_subject|||answers_question|||
    direct_evidence|||quote|||reason
    """
    results = {}

    if not isinstance(response, str):
        return results

    for raw_line in response.splitlines():
        line = raw_line.strip()

        if not line:
            continue

        parts = line.split(
            CRAG_SEPARATOR,
            maxsplit=5
        )

        if len(parts) != 6:
            continue

        (
            index_text,
            same_subject_text,
            answers_question_text,
            direct_evidence_text,
            evidence_quote,
            reason
        ) = parts

        try:
            document_index = int(
                index_text.strip()
            )
        except ValueError:
            continue

        if not (
            0 <= document_index
            < number_of_documents
        ):
            continue

        results[document_index] = {
            "same_subject": parse_crag_boolean(
                same_subject_text
            ),
            "answers_question": parse_crag_boolean(
                answers_question_text
            ),
            "direct_evidence": parse_crag_boolean(
                direct_evidence_text
            ),
            "evidence_quote": evidence_quote.strip(),
            "reason": reason.strip()
        }

    return results
ENABLE_TAVILY_FALLBACK = True

def tavily_fallback(
    query: str,
    max_results: int = 3
):
    contexts = []
    sources = []

    if not TAVILY_API_KEY:
        print(
            "[WARN] Chưa có TAVILY_API_KEY. "
            "Không thể tìm kiếm web."
        )
        return contexts, sources

    try:
        client = TavilyClient(
            api_key=TAVILY_API_KEY
        )

        response = client.search(
            query=query,
            search_depth="advanced",
            max_results=max_results
        )

        for result in response.get("results", []):
            content = str(
                result.get("content", "")
            ).strip()

            if not content:
                continue

            contexts.append(content)

            sources.append({
                "type": "web",
                "url": result.get("url", ""),
                "title": result.get(
                    "title",
                    "Nguồn web"
                ),
                "evidence_chunk": content
            })

    except Exception as error:
        print(
            f"[ERR] Tavily gặp lỗi: {error}"
        )

    return contexts, sources
def crag_grade_and_refine(
    query: str,
    ranked_docs: list,
    max_docs: int = 3,
    max_evidence_chars: int = 3500
):
    valid_contexts = []
    valid_sources = []

    docs_to_grade = list(
        ranked_docs[:max_docs]
    )

    if docs_to_grade:
        formatted_documents = []

        for index, item in enumerate(
            docs_to_grade
        ):
            document_question = str(
                item.get("question", "")
            ).strip()

            # Tương thích ranked_docs cũ.
            if not document_question:
                first_line = str(
                    item.get("doc", "")
                ).splitlines()[0]

                document_question = re.sub(
                    r"^Hỏi:\s*",
                    "",
                    first_line
                ).strip()

            evidence = str(
                item.get(
                    "evidence_chunk",
                    ""
                )
            ).strip()

            evidence = evidence[
                :max_evidence_chars
            ]

            formatted_documents.append(
                f"""
<TAI_LIEU index="{index}">
ID: {item.get("id")}

Câu hỏi hoặc tiêu đề tài liệu:
{document_question}

Evidence:
{evidence}
</TAI_LIEU>
""".strip()
            )

        documents_blob = "\n\n".join(
            formatted_documents
        )

        prompt = f"""
CÂU HỎI NGƯỜI DÙNG:

{query}

CÁC TÀI LIỆU:

{documents_blob}

Đánh giá mỗi tài liệu theo đúng định dạng một dòng đã yêu cầu.
""".strip()

        # QUAN TRỌNG:
        # Không dùng JSON mode nên GPT-OSS không còn lỗi validate JSON.
        response = call_llm(
            prompt,
            system_instruction=(
                STRICT_CRAG_SYSTEM_PROMPT
            ),
            json_mode=False,
            temperature=0.0,
            task_name="crag_grading"
        )

        results_by_index = parse_crag_lines(
            response,
            len(docs_to_grade)
        )

        for index, item in enumerate(
            docs_to_grade
        ):
            grading = results_by_index.get(
                index,
                {
                    "same_subject": False,
                    "answers_question": False,
                    "direct_evidence": False,
                    "evidence_quote": "",
                    "reason": (
                        "CRAG không trả kết quả "
                        "hợp lệ cho tài liệu."
                    )
                }
            )

            evidence = str(
                item.get(
                    "evidence_chunk",
                    ""
                )
            ).strip()

            quote_valid = crag_quote_is_valid(
                grading["evidence_quote"],
                evidence
            )

            accepted = (
                grading["same_subject"]
                and grading["answers_question"]
                and grading["direct_evidence"]
                and quote_valid
            )

            relevance = (
                "relevant"
                if accepted
                else "irrelevant"
            )

            print(
                f"      - Doc {index} "
                f"(ID {item.get('id')}): "
                f"{relevance} — "
                f"{grading['reason']}"
            )

            print(
                "        "
                f"same_subject="
                f"{grading['same_subject']}, "
                f"answers_question="
                f"{grading['answers_question']}, "
                f"direct_evidence="
                f"{grading['direct_evidence']}, "
                f"quote_valid={quote_valid}"
            )

            if not accepted:
                continue

            # Generator chỉ được dùng đúng evidence
            # mà CRAG đã kiểm tra.
            valid_contexts.append(
                evidence
            )

            valid_sources.append({
                "type": "local",
                "id": item.get("id"),
                "score": item.get("score"),
                "question": item.get(
                    "question",
                    ""
                ),
                "evidence_chunk": evidence,
                "support_quote": grading[
                    "evidence_quote"
                ],
                "retrieval_source": item.get(
                    "retrieval_source",
                    ""
                ),
                "crag_reason": grading[
                    "reason"
                ]
            })

    if (
        not valid_contexts
        and ENABLE_TAVILY_FALLBACK
    ):
        print(
            "[CRAG Fallback] Không có tài liệu nội bộ "
            "đủ bằng chứng. Kích hoạt Tavily Search..."
        )

        return tavily_fallback(
            query
        )

    return valid_contexts, valid_sources

# ═══════════════════════════════════════════════════════════════
# MODULE 4: LLM Generator & Safety Guardrail
# Giữ nguyên safety prompt, chỉ bổ sung xử lý lỗi LLM.
# ═══════════════════════════════════════════════════════════════

def generate_safe_answer(query: str, contexts: list, sources: list):
    if not contexts:
        return (
            "Xin lỗi, hệ thống không tìm thấy thông tin y tế đáng tin cậy "
            "để giải đáp câu hỏi của bạn. Vui lòng liên hệ bác sĩ để được tư vấn.",
            False
        )

    context_blob = "\n\n".join(
        [f"[Tài liệu {i + 1}]: {context}" for i, context in enumerate(contexts)]
    )

    # Giữ nguyên prompt safety hiện tại theo yêu cầu của bạn.
    sys_generator = (
        "Bạn là một Bác sĩ AI. Nhiệm vụ: Trả lời câu hỏi người dùng "
        "DỰA VÀO NGỮ CẢNH CUNG CẤP.\n"
        "[SAFETY GUARDRAILS BẮT BUỘC]:\n"
        "1. Không được tự tư vấn thay bác sĩ. Chỉ đưa ra thông tin có trong "
        "ngữ cảnh (chống Hallucination).\n"
        "2. Cuối câu luôn đính kèm: 'Khuyến cáo: Thông tin trên chỉ mang "
        "tính chất tham khảo, hãy đến cơ sở y tế gần nhất để thăm khám kịp thời.'\n"
    )

    prompt = (
        f"NGỮ CẢNH TÀI LIỆU:\n{context_blob}\n\n"
        f"CÂU HỎI:\n{query}\n\n"
        "TRẢ LỜI:"
    )

    final_answer = call_llm(
        prompt,
        system_instruction=sys_generator,
        temperature=0.0,
        task_name="final_generation"
    )

    if not is_valid_llm_output(final_answer):
        return (
            "Xin lỗi, hệ thống đang gặp lỗi khi tạo câu trả lời. "
            "Vui lòng thử lại sau.",
            False
        )

    citations = "\n\n---\n📚 Nguồn Trích Dẫn:\n"
    for idx, src in enumerate(sources):
        if src["type"] == "local":
            citations += (
                f"- [{idx + 1}] CSDL Nội bộ Y khoa "
                f"(ID tài liệu: {src['id']})\n"
            )
        else:
            citations += (
                f"- [{idx + 1}] Thông tin Internet: "
                f"[{src['title']}]({src['url']})\n"
            )

    return final_answer + citations, True


## 6. Chạy Pipeline Hệ thống (Có Cache + Memory)

In [18]:
def _answer_for_memory(answer: str, max_chars=500) -> str:
    """Bỏ phần citation trước khi lưu vào memory để tiết kiệm context."""
    main_answer = str(answer).split("\n\n---\n", 1)[0]
    return main_answer[:max_chars]


def ask_medical_chatbot(
    user_query: str,
    session_id: str = "default",
    verbose: bool = True
):
    user_query = str(user_query).strip()

    if not user_query:
        return {
            "original_query": "",
            "rewritten_query": "",
            "fixed_query": "",
            "answer": "Vui lòng nhập câu hỏi.",
            "sources": [],
            "cache_hit": False,
            "success": False
        }

    current_memory = get_memory(session_id)

    if verbose:
        print("=" * 60)
        print(f"🙋 Người dùng: {user_query}")
        print("=" * 60)

    # Bước 0 phải chạy TRƯỚC answer cache.
    effective_query = rewrite_query_with_history(
        user_query,
        current_memory
    )

    if verbose and effective_query != user_query:
        print(
            f"[0/5] Query Rewriting: "
            f"'{user_query}' → '{effective_query}'"
        )

    # Cache theo câu hỏi đã đầy đủ, không cache theo câu mơ hồ ban đầu.
    cache_key = effective_query.strip().lower()
    cached_result = cache.get(cache.answer_cache, cache_key)

    if cached_result:
        # Tương thích nếu runtime cũ từng cache một chuỗi thay vì dictionary.
        if isinstance(cached_result, str):
            cached_result = {
                "original_query": user_query,
                "rewritten_query": effective_query,
                "fixed_query": effective_query,
                "answer": cached_result,
                "sources": [],
                "cache_hit": True,
                "success": True
            }
        else:
            cached_result = dict(cached_result)
            cached_result["original_query"] = user_query
            cached_result["rewritten_query"] = effective_query
            cached_result["cache_hit"] = True

        # Cache hit vẫn phải đưa lượt nói vào memory.
        current_memory.add("user", user_query)
        current_memory.add(
            "bot",
            _answer_for_memory(cached_result["answer"])
        )

        if verbose:
            print("[CACHE HIT ⚡] Trả kết quả ngay.")
            print("\n🤖 Chatbot Trả Lời:\n")
            print(cached_result["answer"])
            print()

        return cached_result

    if verbose:
        print("[1/5] Query Processing (Sửa lỗi & HyDE)...")

    q_dict = query_processing(effective_query)

    if verbose:
        print(f"      > Đã sửa chính tả: {q_dict['fixed']}")
        print("[2/5 & 3/5] Hybrid Search & Cross-Encoder Reranking...")

    ranked_docs = hybrid_retrieve_and_rerank(q_dict, top_k=3)

    if verbose:
        print("[4/5] CRAG Grading & Knowledge Refinement...")

    contexts, sources = crag_grade_and_refine(
        q_dict["fixed"],
        ranked_docs
    )

    if verbose:
        print("[5/5] Generating Safe Medical Answer...")

    answer, generation_ok = generate_safe_answer(
        q_dict["fixed"],
        contexts,
        sources
    )

    result = {
        "original_query": user_query,
        "rewritten_query": effective_query,
        "fixed_query": q_dict["fixed"],
        "answer": answer,
        "sources": sources,
        "cache_hit": False,
        "success": generation_ok
    }

    # Chỉ cache câu trả lời được sinh thành công.
    if generation_ok:
        cache.set(cache.answer_cache, cache_key, result)

    current_memory.add("user", user_query)
    current_memory.add("bot", _answer_for_memory(answer))

    if verbose:
        print("\n🤖 Chatbot Trả Lời:\n")
        print(answer)
        print()

    return result


## 7. Demo & Test Cases

In [46]:
cache.clear()
session_memories.clear()

print("[OK] Đã xóa cache và toàn bộ session memory cũ.")

[OK] Đã xóa cache và toàn bộ session memory cũ.


In [19]:
TEST_SESSION_ID = "sleep-eye"

#clear_session_memory(TEST_SESSION_ID)

result = ask_medical_chatbot(
    "Có bao nhiêu dây thần kinh trong cơ thể",
    session_id=TEST_SESSION_ID
)

🙋 Người dùng: Có bao nhiêu dây thần kinh trong cơ thể
[CACHE HIT ⚡] Trả kết quả ngay.

🤖 Chatbot Trả Lời:

Số lượng dây thần kinh trong cơ thể con người vẫn chưa có con số chính xác, tuy nhiên theo một số nghiên cứu, con người có tổng cộng 43 cặp dây thần kinh khác nhau. Trong đó, có 12 cặp dây thần kinh trực tiếp kết nối với não, và các cặp còn lại kết nối với tủy sống. Những dây thần kinh này tạo thành một mạng lưới phức tạp, giúp truyền tải thông tin giữa hệ thần kinh trung ương và các bộ phận khác trên cơ thể.

Khuyến cáo: Thông tin trên chỉ mang tính chất tham khảo, hãy đến cơ sở y tế gần nhất để thăm khám kịp thời.

---
📚 **Nguồn Trích Dẫn:**
- [1] CSDL Nội bộ Y khoa (ID tài liệu: 27668)
- [2] CSDL Nội bộ Y khoa (ID tài liệu: 29575)




In [ ]:
ask_medical_chatbot("Triệu chứng của căng cơ đùi")

🙋 Người dùng: Triệu chứng của căng cơ đùi
[1/5] Query Processing (Sửa lỗi & HyDE)...
      > Đã sửa chính tả: Triệu chứng của căng cơ đùi
[2/5 & 3/5] Hybrid Search & Cross-Encoder Reranking...
[4/5] CRAG Grading & Knowledge Refinement...
      - Doc 0 (ID 0): relevant — Evidence trực tiếp liệt kê các triệu chứng của căng cơ đùi.
        same_subject=True, answers_question=True, direct_evidence=True, quote_valid=True
      - Doc 1 (ID 47151): irrelevant — Evidence trực tiếp mô tả các triệu chứng của căng cơ đùi.
        same_subject=True, answers_question=True, direct_evidence=True, quote_valid=False
      - Doc 2 (ID 23170): irrelevant — Evidence
        same_subject=True, answers_question=True, direct_evidence=True, quote_valid=False
[5/5] Generating Safe Medical Answer...

🤖 Chatbot Trả Lời:

Triệu chứng phổ biến của căng cơ đùi bao gồm đau nhức đột ngột tại vùng tổn thương, sưng, bầm tím kèm theo khó khăn khi đi lại, gập duỗi chân hoặc vận động mạnh. Khuyến cáo: Thông tin trên chỉ m

{'original_query': 'Triệu chứng của căng cơ đùi',
 'rewritten_query': 'Triệu chứng của căng cơ đùi',
 'fixed_query': 'Triệu chứng của căng cơ đùi',
 'answer': 'Triệu chứng phổ biến của căng cơ đùi bao gồm đau nhức đột ngột tại vùng tổn thương, sưng, bầm tím kèm theo khó khăn khi đi lại, gập duỗi chân hoặc vận động mạnh. Khuyến cáo: Thông tin trên chỉ mang tính chất tham khảo, hãy đến cơ sở y tế gần nhất để thăm khám kịp thời.\n\n---\n📚 **Nguồn Trích Dẫn:**\n- [1] CSDL Nội bộ Y khoa (ID tài liệu: 0)\n',
 'sources': [{'type': 'local',
   'id': 0,
   'score': 3.524473190307617,
   'question': 'Căng cơ đùi: Nguyên nhân và các phương pháp điều trị',
   'evidence_chunk': 'người bệnh hoạt động quá mức hoặc sai tư thế, đặc biệt trong các môn thể thao như bóng đá, chạy bộ, bóng rổ. Các động tác co duỗi chân đột ngột hoặc kéo dài khi đá bóng, bật nhảy, leo cầu thang cũng có thể làm cơ đùi bị căng. Các động tác co duỗi chân đột ngột hoặc kéo dài khi đá bóng, bật nhảy, leo cầu thang cũng có thể là

In [20]:
ask_medical_chatbot("Căng cơ đùi có trị đc ko")

🙋 Người dùng: Căng cơ đùi có trị đc ko
[1/5] Query Processing (Sửa lỗi & HyDE)...
      > Đã sửa chính tả: Căng cơ đùi có trị được không?
[2/5 & 3/5] Hybrid Search & Cross-Encoder Reranking...
[4/5] CRAG Grading & Knowledge Refinement...
      - Doc 0 (ID 0): relevant — Evidence trực tiếp trả lời đúng chủ đề và ý định.
        same_subject=True, answers_question=True, direct_evidence=True, quote_valid=True
      - Doc 1 (ID 47151): relevant — Evidence trực tiếp trả lời đúng chủ đề và ý định.
        same_subject=True, answers_question=True, direct_evidence=True, quote_valid=True
      - Doc 2 (ID 30821): irrelevant — Tài liệu nói về hút mỡ đùi, không liên quan đến căng cơ.
        same_subject=False, answers_question=False, direct_evidence=False, quote_valid=False
[5/5] Generating Safe Medical Answer...

🤖 Chatbot Trả Lời:

Căng cơ đùi có thể được điều trị hiệu quả tùy vào mức độ tổn thương. Các phương pháp điều trị bao gồm: xoa bóp nhẹ nhàng để thư giãn cơ, chườm đá, nghỉ ngơi, dùng t

{'original_query': 'Căng cơ đùi có trị đc ko',
 'rewritten_query': 'Căng cơ đùi có trị đc ko',
 'fixed_query': 'Căng cơ đùi có trị được không?',
 'answer': 'Căng cơ đùi có thể được điều trị hiệu quả tùy vào mức độ tổn thương. Các phương pháp điều trị bao gồm: xoa bóp nhẹ nhàng để thư giãn cơ, chườm đá, nghỉ ngơi, dùng thuốc giảm đau theo chỉ định bác sĩ, và thực hiện các bài tập vật lý trị liệu. Trong trường hợp nghiêm trọng như rách cơ hoàn toàn, có thể cần phẫu thuật. Bên cạnh đó, việc tập thể dục thường xuyên và phòng ngừa cũng giúp hạn chế nguy cơ tái phát.\n\nKhuyến cáo: Thông tin trên chỉ mang tính chất tham khảo, hãy đến cơ sở y tế gần nhất để thăm khám kịp thời.\n\n---\n📚 Nguồn Trích Dẫn:\n- [1] CSDL Nội bộ Y khoa (ID tài liệu: 0)\n- [2] CSDL Nội bộ Y khoa (ID tài liệu: 47151)\n',
 'sources': [{'type': 'local',
   'id': 0,
   'score': 3.6279985904693604,
   'question': 'Căng cơ đùi: Nguyên nhân và các phương pháp điều trị',
   'evidence_chunk': 'có thể kéo dài trong vài ngày. G

In [ ]:
# ═══ TEST CASE : Query ngoài CSDL (Tavily Fallback) ═══
ask_medical_chatbot("Triệu chứng mới nhất của biến chủng Covid JN.1 năm 2024 là gì?")

🙋 Người dùng: Triệu chứng mới nhất của biến chủng Covid JN.1 năm 2024 là gì?
[1/5] Query Processing (Sửa lỗi & HyDE)...
      > Đã sửa chính tả: Triệu chứng mới nhất của biến chủng Covid JN.1 năm 2024 là gì?
[2/5 & 3/5] Hybrid Search & Cross-Encoder Reranking...
[4/5] CRAG Grading & Knowledge Refinement...
      - Doc 0 (ID 51935): irrelevant — Tài liệu nói về Co vi d - 19 chung chung, không đề cập đến biến chủng JN.1 hay năm 2024.
        same_subject=False, answers_question=False, direct_evidence=False, quote_valid=False
      - Doc 1 (ID 51417): irrelevant — Tài liệu nói về Co vi d - 19 chung chung, không đề cập đến biến chủng JN.1 hay năm 2024.
        same_subject=False, answers_question=False, direct_evidence=False, quote_valid=False
      - Doc 2 (ID 50443): irrelevant — Tài liệu nói về biến thể Delta, không đề cập đến biến chủng JN.1 hay năm 2024.
        same_subject=False, answers_question=False, direct_evidence=False, quote_valid=False
[CRAG Fallback] Không có tài liệu nội b

{'original_query': 'Triệu chứng mới nhất của biến chủng Covid JN.1 năm 2024 là gì?',
 'rewritten_query': 'Triệu chứng mới nhất của biến chủng Covid JN.1 năm 2024 là gì?',
 'fixed_query': 'Triệu chứng mới nhất của biến chủng Covid JN.1 năm 2024 là gì?',
 'answer': 'Triệu chứng mới nhất của biến chủng JN.1 năm 2024 bao gồm các triệu chứng tương tự như các biến thể Omicron khác, cụ thể là:\n\n- Sốt hoặc ớn lạnh  \n- Ho khan  \n- Khó thở hoặc thở gấp (ở những người có nguy cơ cao)  \n- Mệt mỏi  \n- Đau cơ hoặc đau nhức toàn thân  \n- Nhức đầu  \n- Viêm họng  \n- Nghẹt mũi hoặc chảy nước mũi  \n- Buồn nôn hoặc nôn mửa  \n- Tiêu chảy  \n\nNgoài ra, một số bệnh nhân nhiễm JN.1 có thể gặp phải triệu chứng ho khan kéo dài, khàn tiếng hoặc mất giọng, kèm theo đau nhức cơ thể toàn thân. Các triệu chứng này thường nhẹ và dễ nhầm với cảm lạnh thông thường. Tuy nhiên, mức độ nghiêm trọng của bệnh phụ thuộc vào tình trạng sức khỏe và miễn dịch của từng người.\n\nKhuyến cáo: Thông tin trên chỉ mang tí

In [ ]:
# ═══ TEST CASE 4: Multi-turn Conversation Memory ═══
DEMO_SESSION_ID = "demo_multi_turn"
clear_session_memory(DEMO_SESSION_ID)

print("\n" + "🔵"*30 + " DEMO MULTI-TURN " + "🔵"*30 + "\n")

result_1 = ask_medical_chatbot(
    "Triệu chứng đau dạ dày là gì?",
    session_id=DEMO_SESSION_ID
)

result_2 = ask_medical_chatbot(
    "Uống thuốc gì?",
    session_id=DEMO_SESSION_ID
)

result_3 = ask_medical_chatbot(
    "Có tác dụng phụ không?",
    session_id=DEMO_SESSION_ID
)

print("\nCâu hỏi đã viết lại ở lượt 2:", result_2["rewritten_query"])
print("Câu hỏi đã viết lại ở lượt 3:", result_3["rewritten_query"])


In [ ]:
!pip install -q fastapi uvicorn pyngrok nest-asyncio

## 8. Tạo API cho giao diện

**Định nghĩa FastAPI**

In [ ]:
from threading import Lock

from fastapi import FastAPI, HTTPException
from fastapi.encoders import jsonable_encoder
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field


app = FastAPI(
    title="Medical RAG Chatbot API",
    version="1.0.0",
)


# Tạm thời cho phép frontend bất kỳ gọi API.
# Vì không dùng cookie nên để allow_credentials=False.
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)


# Chỉ cho một request chạy pipeline GPU tại một thời điểm.
# Tránh nhiều request đồng thời làm tràn VRAM hoặc xung đột model.
pipeline_lock = Lock()


class ChatRequest(BaseModel):
    query: str = Field(
        min_length=1,
        max_length=2000,
    )

    # Nên bắt buộc frontend gửi session_id,
    # tránh tất cả người dùng dùng chung memory "default".
    session_id: str = Field(
        min_length=1,
        max_length=200,
    )


@app.get("/")
def root():
    return {
        "service": "medical-rag-chatbot",
        "status": "running",
        "docs": "/docs",
    }


@app.get("/health")
def health_check():
    required_objects = {
        "embedding_fn": "embedding_fn" in globals(),
        "collection": "collection" in globals(),
        "bm25": "bm25" in globals(),
        "reranker": "reranker" in globals(),
        "chatbot": "ask_medical_chatbot" in globals(),
    }

    chroma_count = 0

    if required_objects["collection"]:
        try:
            chroma_count = collection.count()
        except Exception:
            chroma_count = 0

    ready = (
        all(required_objects.values())
        and chroma_count > 0
    )

    return {
        "status": "ok" if ready else "not_ready",
        "ready": ready,
        "components": required_objects,
        "chroma_count": chroma_count,
    }


@app.post("/api/chat")
def chat(payload: ChatRequest):
    try:
        # Tránh hai request cùng chạy BGE/reranker trên GPU.
        with pipeline_lock:
            result = ask_medical_chatbot(
                payload.query,
                session_id=payload.session_id,
                verbose=False,
            )

        # Chuyển numpy types hoặc object đặc biệt về JSON hợp lệ.
        return JSONResponse(
            content=jsonable_encoder(result)
        )

    except Exception as error:
        print(
            "[API ERROR]",
            type(error).__name__,
            str(error),
        )

        raise HTTPException(
            status_code=500,
            detail=str(error),
        ) from error


print("[OK] Đã định nghĩa FastAPI app.")

[OK] Đã định nghĩa FastAPI app.


**Khởi động Uvicorn**

In [ ]:
import asyncio
import socket
import threading
import time

import uvicorn


API_HOST = "127.0.0.1"
API_PORT = 8000


def port_is_open(
    host: str = API_HOST,
    port: int = API_PORT,
) -> bool:
    try:
        with socket.create_connection(
            (host, port),
            timeout=1,
        ):
            return True
    except OSError:
        return False


def run_api_server():
    """
    Chạy Uvicorn trong một event loop riêng.

    Không gọi uvicorn.run() hay Server.run(),
    nên tránh asyncio.run() đã bị nest_asyncio patch.
    """
    global _api_server
    global _api_event_loop

    _api_event_loop = asyncio.new_event_loop()
    asyncio.set_event_loop(_api_event_loop)

    config = uvicorn.Config(
        app=app,
        host=API_HOST,
        port=API_PORT,
        loop="asyncio",
        log_level="info",
        access_log=True,
    )

    _api_server = uvicorn.Server(config)

    try:
        _api_event_loop.run_until_complete(
            _api_server.serve()
        )
    finally:
        _api_event_loop.close()


# Dừng server cũ do notebook tạo nếu có.
if "_api_server" in globals():
    try:
        _api_server.should_exit = True
        time.sleep(2)
    except Exception:
        pass


if port_is_open():
    print(
        "[OK] FastAPI đã chạy sẵn tại "
        "http://127.0.0.1:8000"
    )

else:
    _api_thread = threading.Thread(
        target=run_api_server,
        daemon=True,
        name="medical-rag-api",
    )

    _api_thread.start()

    # Chờ tối đa 30 giây để server mở port.
    for _ in range(60):
        if port_is_open():
            break

        time.sleep(0.5)

    if not port_is_open():
        raise RuntimeError(
            "Uvicorn vẫn không mở được port 8000. "
            "Xem log phía trên để tìm lỗi trong FastAPI app."
        )

    print(
        "[OK] FastAPI thực sự đang chạy tại "
        "http://127.0.0.1:8000"
    )

INFO:     Started server process [5190]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


[OK] FastAPI thực sự đang chạy tại http://127.0.0.1:8000


**Test FastAPI local trước**

In [ ]:
import requests


local_health_url = (
    "http://127.0.0.1:8000/health"
)

response = requests.get(
    local_health_url,
    timeout=30,
)

print("Status:", response.status_code)
print("Content-Type:", response.headers.get("content-type"))
print(response.text)

INFO:     127.0.0.1:41786 - "GET /health HTTP/1.1" 200 OK
Status: 200
Content-Type: application/json
{"status":"ok","ready":true,"components":{"embedding_fn":true,"collection":true,"bm25":true,"reranker":true,"chatbot":true},"chroma_count":413344}


**Test chatbot qua local API**

In [ ]:
local_response = requests.post(
    "http://127.0.0.1:8000/api/chat",
    json={
        "query": "Hiện tượng ngủ mở mắt là gì?",
        "session_id": "local-api-test-001",
    },
    timeout=300,
)

print("Status:", local_response.status_code)
print("Content-Type:", local_response.headers.get("content-type"))

if (
    "application/json"
    in local_response.headers.get(
        "content-type",
        "",
    )
):
    print(local_response.json())
else:
    print(local_response.text)

      - Doc 0 (ID 40514): irrelevant — Evidence trực tiếp trả lời đúng chủ đề và ý định.
        same_subject=True, answers_question=True, direct_evidence=True, quote_valid=False
      - Doc 1 (ID 40516): relevant — Evidence trực tiếp trả lời đúng chủ đề và ý định.
        same_subject=True, answers_question=True, direct_evidence=True, quote_valid=True
      - Doc 2 (ID 40464): relevant — Evidence trực tiếp trả lời đúng chủ đề và ý định.
        same_subject=True, answers_question=True, direct_evidence=True, quote_valid=True
INFO:     127.0.0.1:54706 - "POST /api/chat HTTP/1.1" 200 OK
Status: 200
Content-Type: application/json
{'original_query': 'Hiện tượng ngủ mở mắt là gì?', 'rewritten_query': 'Hiện tượng ngủ mở mắt là gì?', 'fixed_query': 'Hiện tượng ngủ mở mắt là gì?', 'answer': 'Hiện tượng ngủ mở mắt là tình trạng người bệnh ngủ nhưng mí mắt không có khả năng khép kín hoàn toàn. Thậm chí, kể cả khi bệnh nhân đã chủ động nhắm mắt vẫn không thể khép kín mắt. Đây là một hiện tượng kh

**Xác thực và tạo ngrok tunnel**

In [ ]:
from google.colab import userdata
from pyngrok import ngrok


NGROK_AUTHTOKEN = userdata.get(
    "NGROK_AUTHTOKEN"
)

if not NGROK_AUTHTOKEN:
    raise ValueError(
        "Không tìm thấy NGROK_AUTHTOKEN "
        "trong Colab Secrets."
    )


ngrok.set_auth_token(
    NGROK_AUTHTOKEN
)

# Đóng các tunnel ngrok cũ,
# không ảnh hưởng đến FastAPI.
ngrok.kill()


# Quan trọng:
# dùng 127.0.0.1 thay vì chỉ ghi 8000/localhost,
# tránh ngrok thử kết nối IPv6 [::1].
public_tunnel = ngrok.connect(
    addr="127.0.0.1:8000",
    proto="http",
)

public_url = public_tunnel.public_url


print("Public URL:", public_url)
print("Health:", f"{public_url}/health")
print("Chat API:", f"{public_url}/api/chat")
print("Swagger:", f"{public_url}/docs")

Public URL: https://steadier-swerve-handoff.ngrok-free.dev
Health: https://steadier-swerve-handoff.ngrok-free.dev/health
Chat API: https://steadier-swerve-handoff.ngrok-free.dev/api/chat
Swagger: https://steadier-swerve-handoff.ngrok-free.dev/docs


In [ ]:
from google.colab import userdata
from pyngrok import ngrok
import requests


NGROK_AUTHTOKEN = userdata.get(
    "NGROK_AUTHTOKEN"
)

if not NGROK_AUTHTOKEN:
    raise ValueError(
        "Không tìm thấy NGROK_AUTHTOKEN "
        "trong Colab Secrets."
    )


# Domain cố định được cấp trong ngrok Dashboard.
NGROK_FIXED_URL = (
    "https://steadier-swerve-handoff.ngrok-free.dev"
)


# FastAPI phải chạy trước tại:
# http://127.0.0.1:8000
local_health = requests.get(
    "http://127.0.0.1:8000/health",
    timeout=30,
)

if local_health.status_code != 200:
    raise RuntimeError(
        "FastAPI local chưa hoạt động."
    )


ngrok.set_auth_token(
    NGROK_AUTHTOKEN
)

# Dừng tunnel cũ trong runtime hiện tại.
ngrok.kill()


# Khởi động tunnel mới nhưng sử dụng lại domain cố định.
public_tunnel = ngrok.connect(
    addr="127.0.0.1:8000",
    proto="http",
    url=NGROK_FIXED_URL,
)

public_url = public_tunnel.public_url


print("Public URL:", public_url)
print("Health:", f"{public_url}/health")
print("Chat API:", f"{public_url}/api/chat")
print("Swagger:", f"{public_url}/docs")

INFO:     127.0.0.1:48662 - "GET /health HTTP/1.1" 200 OK
Public URL: https://steadier-swerve-handoff.ngrok-free.dev
Health: https://steadier-swerve-handoff.ngrok-free.dev/health
Chat API: https://steadier-swerve-handoff.ngrok-free.dev/api/chat
Swagger: https://steadier-swerve-handoff.ngrok-free.dev/docs


**Test ngrok**

In [ ]:
ngrok_headers = {
    "ngrok-skip-browser-warning": "true"
}


public_health_response = requests.get(
    f"{public_url}/health",
    headers=ngrok_headers,
    timeout=30,
)

print(
    "Status:",
    public_health_response.status_code,
)

print(
    "Content-Type:",
    public_health_response.headers.get(
        "content-type"
    ),
)

print(public_health_response.text)

INFO:     34.126.72.88:0 - "GET /health HTTP/1.1" 200 OK
Status: 200
Content-Type: application/json
{"status":"ok","ready":true,"components":{"embedding_fn":true,"collection":true,"bm25":true,"reranker":true,"chatbot":true},"chroma_count":413344}


In [ ]:
public_chat_response = requests.post(
    f"{public_url}/api/chat",
    headers=ngrok_headers,
    json={
        "query": "Hiện tượng ngủ mở mắt là gì?",
        "session_id": "ngrok-api-test-001",
    },
    timeout=300,
)

print(
    "Status:",
    public_chat_response.status_code,
)

print(
    "Content-Type:",
    public_chat_response.headers.get(
        "content-type"
    ),
)

if (
    "application/json"
    in public_chat_response.headers.get(
        "content-type",
        "",
    )
):
    print(public_chat_response.json())
else:
    print(public_chat_response.text)

INFO:     34.126.72.88:0 - "POST /api/chat HTTP/1.1" 200 OK
Status: 200
Content-Type: application/json
{'original_query': 'Hiện tượng ngủ mở mắt là gì?', 'rewritten_query': 'Hiện tượng ngủ mở mắt là gì?', 'fixed_query': 'Hiện tượng ngủ mở mắt là gì?', 'answer': 'Hiện tượng ngủ mở mắt là tình trạng người bệnh ngủ nhưng mí mắt không có khả năng khép kín hoàn toàn. Thậm chí, kể cả khi bệnh nhân đã chủ động nhắm mắt vẫn không thể khép kín mắt. Đây là một hiện tượng không bình thường và có thể là biểu hiện của một số bệnh lý liên quan đến mắt hoặc dây thần kinh.\n\nKhuyến cáo: Thông tin trên chỉ mang tính chất tham khảo, hãy đến cơ sở y tế gần nhất để thăm khám kịp thời.\n\n---\n📚 **Nguồn Trích Dẫn:**\n- [1] CSDL Nội bộ Y khoa (ID tài liệu: 40516)\n- [2] CSDL Nội bộ Y khoa (ID tài liệu: 40464)\n', 'sources': [{'type': 'local', 'id': 40516, 'score': 3.685432195663452, 'question': 'Vì sao ngủ mở mắt? Cách trị ngủ mở mắt', 'evidence_chunk': 'Thông thường, khi ngủ mắt chúng ta sẽ nhắm lại hoàn 

## 9. Metric

In [21]:
%pip install -q rouge-score bert-score underthesea

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 154.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 94.2 MB/s eta 0:00:00


**Tạo tập 10 mẫu đầu tiên**

In [50]:


# Kiểm tra các thành phần của notebook đã sẵn sàng.
required_objects = [
    "parent_documents",
    "ask_medical_chatbot",
    "cache",
    "session_memories",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Chưa chạy đủ các cell RAG. Thiếu:\n"
        + "\n".join(missing_objects)
    )


# Lấy đúng 10 parent document đầu tiên theo ID.
first_10_ids = sorted(
    parent_documents.keys()
)[:11]

test_records = []

for parent_id in first_10_ids:
    document = parent_documents[parent_id]

    test_records.append({
        "sample_id": int(parent_id),
        "question": str(
            document["question"]
        ).strip(),
        "reference_answer": str(
            document["answer"]
        ).strip(),
    })


rag_test_df = pd.DataFrame(test_records)

print("[OK] Số mẫu test:", len(rag_test_df))
print("[OK] Các cột:", rag_test_df.columns.tolist())

display(rag_test_df.head(5))

[OK] Số mẫu test: 11
[OK] Các cột: ['sample_id', 'question', 'reference_answer']


,sample_id,question,reference_answer
0,0,Căng cơ đùi: Nguyên nhân và các phương pháp đi...,Căng cơ đùi có thể coi là một trong những chấn...
1,1,Chuột rút cơ lưng: Nguyên nhân và biện pháp ph...,Chuột rút cơ lưng có thể gây đau đớn và ảnh hư...
2,2,Gãy xương đòn bao lâu thì tháo nẹp và những lư...,Gãy xương đòn bao lâu thì tháo nẹp là câu hỏi ...
3,3,Cách chữa viêm khớp cùng chậu tại nhà: Phương ...,Cách chữa viêm khớp cùng chậu tại nhà như bài ...
4,4,Gãy xương cẳng tay bao lâu thì lành và những đ...,Gãy xương cẳng tay bao lâu thì lành là điều mà...


**Hàm làm sạch câu trả lời RAG**

In [51]:
import re
import unicodedata


SAFETY_DISCLAIMER = (
    "Khuyến cáo: Thông tin trên chỉ mang "
    "tính chất tham khảo, hãy đến cơ sở y tế "
    "gần nhất để thăm khám kịp thời."
)


def normalize_text(text: str) -> str:
    """Chuẩn hóa nhẹ, không làm thay đổi nội dung."""
    text = unicodedata.normalize(
        "NFC",
        str(text),
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text.strip()


def clean_rag_answer(answer: str) -> str:
    """
    Loại:
    - phần nguồn trích dẫn;
    - câu khuyến cáo cố định;
    - khoảng trắng thừa.
    """
    answer = str(answer)

    # Notebook hiện tại dùng separator này.
    answer = answer.split(
        "\n\n---\n",
        maxsplit=1,
    )[0]

    # Phòng trường hợp format nguồn thay đổi.
    citation_markers = [
        "📚 **Nguồn Trích Dẫn:**",
        "📚 Nguồn Trích Dẫn:",
        "Nguồn tham khảo:",
        "Tài liệu tham khảo:",
    ]

    for marker in citation_markers:
        if marker in answer:
            answer = answer.split(
                marker,
                maxsplit=1,
            )[0]

    # Loại câu disclaimer giống nhau ở mọi dự đoán.
    answer = answer.replace(
        SAFETY_DISCLAIMER,
        "",
    )

    return normalize_text(answer)


# Kiểm tra nhanh.
example = (
    "Đây là câu trả lời.\n\n"
    + SAFETY_DISCLAIMER
    + "\n\n---\n📚 **Nguồn Trích Dẫn:**\n- [1] ..."
)

print(clean_rag_answer(example))

Đây là câu trả lời.


**Chạy RAG trên 10 mẫu**

In [52]:
import json
import time
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm


RAG_PREDICTION_PATH = Path(
    "/content/rag_first_100_predictions.csv"
)

# Nghỉ ngắn giữa các query để giảm nguy cơ rate limit.
PAUSE_SECONDS = 1.5

# Thử lại khi pipeline lỗi tạm thời.
MAX_RETRIES = 3


# ============================================================
# 1. Load checkpoint cũ nếu đã chạy dở
# ============================================================

if RAG_PREDICTION_PATH.exists():
    old_results_df = pd.read_csv(
        RAG_PREDICTION_PATH
    )

    completed_ids = set(
        old_results_df["sample_id"]
        .astype(int)
        .tolist()
    )

    results = old_results_df.to_dict(
        orient="records"
    )

    print(
        f"[RESUME] Đã có {len(completed_ids)} "
        "mẫu trong checkpoint."
    )

else:
    completed_ids = set()
    results = []


# ============================================================
# 2. Xóa cache/memory cũ
# ============================================================

cache.clear()
session_memories.clear()

print("[OK] Đã xóa cache và conversation memory cũ.")


# ============================================================
# 3. Tắt Tavily trong phép đo nội bộ
# ============================================================

old_tavily_setting = ENABLE_TAVILY_FALLBACK
ENABLE_TAVILY_FALLBACK = False

print(
    "[INFO] Tavily fallback đã tắt "
    "trong quá trình benchmark."
)


# ============================================================
# 4. Chạy 100 mẫu
# ============================================================

try:
    for _, row in tqdm(
        rag_test_df.iterrows(),
        total=len(rag_test_df),
        desc="Đánh giá RAG",
    ):
        sample_id = int(row["sample_id"])

        if sample_id in completed_ids:
            continue

        question = str(row["question"])
        reference = str(
            row["reference_answer"]
        )

        prediction = ""
        success = False
        error_message = ""
        sources = []
        elapsed_seconds = None

        for attempt in range(
            1,
            MAX_RETRIES + 1,
        ):
            # Mỗi mẫu là một cuộc hội thoại độc lập.
            session_id = (
                f"rag-evaluation-{sample_id}-"
                f"attempt-{attempt}"
            )

            try:
                started_at = time.perf_counter()

                result = ask_medical_chatbot(
                    question,
                    session_id=session_id,
                    verbose=False,
                )

                elapsed_seconds = (
                    time.perf_counter()
                    - started_at
                )

                if not isinstance(result, dict):
                    raise TypeError(
                        "ask_medical_chatbot không trả dictionary."
                    )

                raw_answer = result.get(
                    "answer",
                    "",
                )

                prediction = clean_rag_answer(
                    raw_answer
                )

                sources = result.get(
                    "sources",
                    [],
                )

                success = bool(
                    result.get("success", False)
                    and prediction
                )

                if success:
                    error_message = ""
                    break

                error_message = (
                    "Pipeline trả success=False "
                    "hoặc câu trả lời rỗng."
                )

            except Exception as error:
                error_message = (
                    f"{type(error).__name__}: {error}"
                )

            print(
                f"[WARN] Sample {sample_id}, "
                f"lần {attempt}/{MAX_RETRIES}: "
                f"{error_message}"
            )

            # Lần sau chờ lâu hơn lần trước.
            time.sleep(5 * attempt)

        record = {
            "sample_id": sample_id,
            "question": question,
            "reference_answer": reference,
            "rag_prediction": prediction,
            "success": success,
            "latency_seconds": elapsed_seconds,
            "num_sources": len(sources),
            "source_types": json.dumps(
                [
                    source.get("type", "")
                    for source in sources
                ],
                ensure_ascii=False,
            ),
            "error": error_message,
        }

        results.append(record)
        completed_ids.add(sample_id)

        # Lưu ngay sau từng mẫu.
        result_df = pd.DataFrame(results)

        result_df.to_csv(
            RAG_PREDICTION_PATH,
            index=False,
            encoding="utf-8-sig",
        )

        time.sleep(PAUSE_SECONDS)

finally:
    # Trả lại cấu hình Tavily ban đầu.
    ENABLE_TAVILY_FALLBACK = (
        old_tavily_setting
    )


rag_predictions_df = pd.DataFrame(results)

rag_predictions_df = (
    rag_predictions_df
    .sort_values("sample_id")
    .reset_index(drop=True)
)

print("\n[OK] Hoàn thành.")
print("[OK] File:", RAG_PREDICTION_PATH)
print(
    "[OK] Thành công:",
    int(rag_predictions_df["success"].sum()),
    "/",
    len(rag_predictions_df),
)

display(
    rag_predictions_df[
        [
            "sample_id",
            "question",
            "success",
            "latency_seconds",
            "num_sources",
        ]
    ].head(10)
)

[RESUME] Đã có 10 mẫu trong checkpoint.
[OK] Đã xóa cache và conversation memory cũ.
[INFO] Tavily fallback đã tắt trong quá trình benchmark.


Đánh giá RAG:   0%|          | 0/11 [00:00<?, ?it/s]

      - Doc 0 (ID 10): relevant — Tài liệu trực tiếp liệt kê các bài tập yoga phổ biến cho người đau lưng.
        same_subject=True, answers_question=True, direct_evidence=True, quote_valid=True
      - Doc 1 (ID 45231): irrelevant — Tài liệu nói về gù lưng ở trẻ, không liên quan đến bài tập yoga cho người đau lưng.
        same_subject=False, answers_question=False, direct_evidence=False, quote_valid=False

[OK] Hoàn thành.
[OK] File: /content/rag_first_100_predictions.csv
[OK] Thành công: 10 / 11


,sample_id,question,success,latency_seconds,num_sources
0,0,Căng cơ đùi: Nguyên nhân và các phương pháp đi...,True,6.048168,2
1,1,Chuột rút cơ lưng: Nguyên nhân và biện pháp ph...,True,24.788014,2
2,2,Gãy xương đòn bao lâu thì tháo nẹp và những lư...,True,6.917439,2
3,3,Cách chữa viêm khớp cùng chậu tại nhà: Phương ...,True,36.705683,1
4,4,Gãy xương cẳng tay bao lâu thì lành và những đ...,True,44.717355,2
5,5,Gãy xương cụt bao lâu thì lành và làm sao điều...,True,6.787250,1
6,6,Đau thần kinh tọa có chữa hết không và cách đi...,True,27.405286,2
7,7,Ngâm chân chữa đau thần kinh tọa: Cách thực hi...,True,43.686507,1
8,8,Bài tập phục hồi chức năng sau mổ cột sống và ...,True,33.769971,1
9,9,Phải làm gì khi bị đau lưng dưới dai dẳng? Nhữ...,False,23.235501,0


**Kiểm tra số mẫu thành công**

In [53]:
import pandas as pd


rag_predictions_df = pd.read_csv(
    "/content/rag_first_100_predictions.csv"
)

evaluation_df = rag_predictions_df[
    rag_predictions_df["success"] == True
].copy()

evaluation_df["rag_prediction"] = (
    evaluation_df["rag_prediction"]
    .fillna("")
    .astype(str)
    .str.strip()
)

evaluation_df["reference_answer"] = (
    evaluation_df["reference_answer"]
    .fillna("")
    .astype(str)
    .str.strip()
)

evaluation_df = evaluation_df[
    (evaluation_df["rag_prediction"] != "")
    & (evaluation_df["reference_answer"] != "")
].reset_index(drop=True)

print(
    "[INFO] Số mẫu dùng tính metric:",
    len(evaluation_df),
    "/",
    len(rag_predictions_df),
)

if len(evaluation_df) == 0:
    raise RuntimeError(
        "Không có mẫu hợp lệ để tính metric."
    )

[INFO] Số mẫu dùng tính metric: 10 / 11


**Tính ROUGE-L**

In [54]:
import numpy as np

from underthesea import word_tokenize
from rouge_score import rouge_scorer


def prepare_vietnamese_for_rouge(
    text: str,
) -> str:
    text = normalize_text(text).lower()

    # Ví dụ: "đau dạ dày" có thể thành "đau_dạ_dày".
    return word_tokenize(
        text,
        format="text",
    )


scorer = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=False,
)


rouge_l_precision = []
rouge_l_recall = []
rouge_l_f1 = []


for _, row in tqdm(
    evaluation_df.iterrows(),
    total=len(evaluation_df),
    desc="Tính ROUGE-L",
):
    prediction = (
        prepare_vietnamese_for_rouge(
            row["rag_prediction"]
        )
    )

    reference = (
        prepare_vietnamese_for_rouge(
            row["reference_answer"]
        )
    )

    score = scorer.score(
        target=reference,
        prediction=prediction,
    )["rougeL"]

    rouge_l_precision.append(
        score.precision
    )

    rouge_l_recall.append(
        score.recall
    )

    rouge_l_f1.append(
        score.fmeasure
    )


evaluation_df["rouge_l_precision"] = (
    rouge_l_precision
)

evaluation_df["rouge_l_recall"] = (
    rouge_l_recall
)

evaluation_df["rouge_l_f1"] = (
    rouge_l_f1
)


print(
    "[RESULT] ROUGE-L Precision:",
    f"{np.mean(rouge_l_precision):.4f}",
)

print(
    "[RESULT] ROUGE-L Recall:",
    f"{np.mean(rouge_l_recall):.4f}",
)

print(
    "[RESULT] ROUGE-L F1:",
    f"{np.mean(rouge_l_f1):.4f}",
)

Tính ROUGE-L:   0%|          | 0/10 [00:00<?, ?it/s]

[RESULT] ROUGE-L Precision: 0.8187
[RESULT] ROUGE-L Recall: 0.1000
[RESULT] ROUGE-L F1: 0.1756


**Tính BERTScore**

In [55]:
import numpy as np
import torch

from bert_score import score as bert_score


predictions = (
    evaluation_df["rag_prediction"]
    .map(normalize_text)
    .tolist()
)

references = (
    evaluation_df["reference_answer"]
    .map(normalize_text)
    .tolist()
)


BERTSCORE_DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "[INFO] BERTScore device:",
    BERTSCORE_DEVICE,
)


bert_precision, bert_recall, bert_f1 = (
    bert_score(
        cands=predictions,
        refs=references,

        # Tự chọn multilingual backbone
        # phù hợp với tiếng Việt.
        lang="vi",

        device=BERTSCORE_DEVICE,

        # Giảm nếu thiếu VRAM.
        batch_size=4,

        verbose=True,

        # Không dùng baseline rescaling
        # để tránh phụ thuộc baseline theo ngôn ngữ.
        rescale_with_baseline=False,
    )
)


evaluation_df[
    "bertscore_precision"
] = bert_precision.cpu().numpy()

evaluation_df[
    "bertscore_recall"
] = bert_recall.cpu().numpy()

evaluation_df[
    "bertscore_f1"
] = bert_f1.cpu().numpy()


print(
    "[RESULT] BERTScore Precision:",
    f"{evaluation_df['bertscore_precision'].mean():.4f}",
)

print(
    "[RESULT] BERTScore Recall:",
    f"{evaluation_df['bertscore_recall'].mean():.4f}",
)

print(
    "[RESULT] BERTScore F1:",
    f"{evaluation_df['bertscore_f1'].mean():.4f}",
)

[INFO] BERTScore device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/5 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/3 [00:00<?, ?it/s]

done in 0.23 seconds, 43.62 sentences/sec
[RESULT] BERTScore Precision: 0.7125
[RESULT] BERTScore Recall: 0.6692
[RESULT] BERTScore F1: 0.6901


**Tổng hợp kết quả**

In [56]:
import numpy as np
import pandas as pd


summary = {
    "model": "RAG",
    "requested_samples": 100,
    "successful_samples": len(evaluation_df),

    "rouge_l_precision_mean": float(
        evaluation_df[
            "rouge_l_precision"
        ].mean()
    ),

    "rouge_l_recall_mean": float(
        evaluation_df[
            "rouge_l_recall"
        ].mean()
    ),

    "rouge_l_f1_mean": float(
        evaluation_df[
            "rouge_l_f1"
        ].mean()
    ),

    "rouge_l_f1_std": float(
        evaluation_df[
            "rouge_l_f1"
        ].std(ddof=1)
    ),

    "bertscore_precision_mean": float(
        evaluation_df[
            "bertscore_precision"
        ].mean()
    ),

    "bertscore_recall_mean": float(
        evaluation_df[
            "bertscore_recall"
        ].mean()
    ),

    "bertscore_f1_mean": float(
        evaluation_df[
            "bertscore_f1"
        ].mean()
    ),

    "bertscore_f1_std": float(
        evaluation_df[
            "bertscore_f1"
        ].std(ddof=1)
    ),

    "latency_mean_seconds": float(
        evaluation_df[
            "latency_seconds"
        ].mean()
    ),

    "success_rate": float(
        rag_predictions_df[
            "success"
        ].mean()
    ),
}


summary_df = pd.DataFrame([summary])

display(summary_df.T)

,0
model,RAG
requested_samples,100
successful_samples,10
rouge_l_precision_mean,0.818675
rouge_l_recall_mean,0.100026
rouge_l_f1_mean,0.175635
rouge_l_f1_std,0.049638
bertscore_precision_mean,0.712488
bertscore_recall_mean,0.669154
bertscore_f1_mean,0.690054


In [57]:
report_table = pd.DataFrame({
    "Mô hình": ["RAG"],
    "Số mẫu": [
        len(evaluation_df)
    ],
    "ROUGE-L F1": [
        (
            f"{evaluation_df['rouge_l_f1'].mean():.4f}"
            f" ± "
            f"{evaluation_df['rouge_l_f1'].std(ddof=1):.4f}"
        )
    ],
    "BERTScore F1": [
        (
            f"{evaluation_df['bertscore_f1'].mean():.4f}"
            f" ± "
            f"{evaluation_df['bertscore_f1'].std(ddof=1):.4f}"
        )
    ],
    "Latency trung bình (giây)": [
        round(
            evaluation_df[
                "latency_seconds"
            ].mean(),
            2,
        )
    ],
    "Tỷ lệ thành công": [
        f"{rag_predictions_df['success'].mean() * 100:.2f}%"
    ],
})

display(report_table)

,Mô hình,Số mẫu,ROUGE-L F1,BERTScore F1,Latency trung bình (giây),Tỷ lệ thành công
0,RAG,10,0.1756 ± 0.0496,0.6901 ± 0.0212,23.74,90.91%
